# Claude Sonnet 4.6 Full Run

In [0]:
import pandas as pd
manual_score = pd.read_csv("/Workspace/Users/user/user/manual_score.csv")
display(manual_score)

notes_enrolled4 = pd.read_csv("/Workspace/Users/user/user/notes_enrolled4.csv")
display(notes_enrolled4)

checklist_filtered = pd.read_csv("/Workspace/Users/user/user/checklist_filtered.csv")
display(checklist_filtered)

In [0]:
dbutils.secrets.listScopes()

In [0]:
dbutils.secrets.list("my-agent-secrets")

In [0]:
%pip install -U anthropic

In [0]:
import openai
import pandas as pd
from openai import AzureOpenAI

import os
import pandas as pd
import numpy as np
import time
import json
import ast
import copy
from IPython.display import IFrame
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
import re


In [0]:
import importlib.metadata as m
for p in ("anthropic", "pydantic", "pydantic_core", "typing_extensions"):
    print(p, m.version(p))

In [0]:
import typing_extensions; print(typing_extensions.__file__)

In [0]:
import importlib.metadata as md
import sys
import importlib.util

print("python:", sys.version)
print("anthropic package version:", md.version("anthropic"))
print("typing_extensions version:", md.version("typing_extensions"))
print("typing_extensions path:", importlib.util.find_spec("typing_extensions").origin)
print("typing path:", importlib.util.find_spec("typing").origin)

In [0]:
import sys
print(sys.version)

In [0]:
%pip show anthropic

In [0]:
from anthropic import AnthropicFoundry

api_key = dbutils.secrets.get(scope="my-agent-secrets", key="sonnet-api-key")
endpoint = dbutils.secrets.get(scope="my-agent-secrets", key="sonnet-endpoint")

client = AnthropicFoundry(
    api_key=api_key,
    base_url=endpoint + "/anthropic"
)

with client.messages.stream(
    model="claude-sonnet-4-6",
    max_tokens=128000,
    messages=[{"role": "user", "content": "Say hello"}]
) as stream:
    response_text = ""
    for text in stream.text_stream:
        response_text += text

print(response_text)

In [0]:
# Test 1: Check your client config
print(f"API Key set: {bool(client.api_key)}")
print(f"API Key (last 4): ...{client.api_key[-4:]}")
print(f"Base URL: {client.base_url}")
# print(f"API Version: {client._api_version}")

In [0]:
# phenotype-specific justification
import json

phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)


phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)

system_prompt = f"""You are a world class board-certified medical geneticist performing a structured chart review.

TASK: Read the provided clinical note and determine whether each phenotype below is present (1) or absent (0) for the patient.

RULES:
- Output ONLY a valid JSON object and nothing else.
- The JSON must contain exactly one key: "Phenotypes".
- "Phenotypes": An object where each key is a phenotype name exactly as listed below, and each value is an object with three keys:
  - "score": 0 or 1
  - "justification": Why THIS SPECIFIC phenotype is present or absent. Do not summarize the entire note. Explain what evidence in the note directly relates to this phenotype, or state that no relevant evidence was found.
  - "quote": The exact word-for-word excerpt from the note that supports your decision for THIS SPECIFIC phenotype. Copy the text exactly as written in the note. Use "" if no relevant text exists.
- Each phenotype's justification and quote must pertain ONLY to that phenotype. Do not reuse the same justification or quote across multiple phenotypes unless the exact same sentence genuinely supports both.

SCORING CRITERIA:
- Use 1 ONLY if the note provides clear evidence that the patient HAS the phenotype as a clinically significant finding — a confirmed diagnosis, an objective measurement that meets a clinical threshold for pathology, or symptoms explicitly identified as pathological by the clinician.
- Use 0 if the note does not mention the phenotype, explicitly rules it out, there is insufficient evidence, or a reported value does not meet the accepted clinical threshold for that phenotype.
- If the note mentions a phenotype was historically present but has since resolved, still code as 1.
- Score based ONLY on findings documented for THIS patient. If the note describes features generally associated with a genotype, syndrome, or condition but does not confirm that this specific patient exhibits them, do not score those features as present.

CONTINUOUS AND QUANTITATIVE FEATURES:
- For any phenotype involving a measurable value, do not score based on being merely non-average.
- Score 1 ONLY if the value meets or exceeds the accepted clinical threshold for that phenotype based on CDC and WHO guidelines. For growth parameters, this generally means at or beyond the 3rd/97th percentile (±2 SD) for age and sex.
- If the note provides a raw value with the patient's age and sex, assess whether it crosses the clinical threshold. If the note already states a percentile, use that directly.
- If a value crosses the clinical threshold only marginally — sitting just past the cutoff or hovering at the borderline — do NOT score it 1 on the strength of a single reading. In that borderline case, score 1 only if the note documents a history of the finding: it has been observed on more than one occasion, or is described as persistent, recurrent, chronic, or longstanding. A lone borderline reading with no such documented history should be scored 0. Values that fall clearly and substantially beyond the threshold remain 1 regardless of whether a prior history is noted.
- A value that is below or above average but still within the normal range does NOT qualify as a phenotype. A developmental milestone that is slightly late but within the accepted normal window does NOT qualify as delayed.
- State the threshold you applied in your justification so your reasoning is transparent.

DETERMINING AGE AT A DEVELOPMENTAL MILESTONE:
- Before scoring any developmental delay, establish the patient's age at the milestone, then compare it to that milestone's threshold age.
- If the note records when the milestone occurred, use that. When it gives a date rather than an age, derive the age at the milestone from the age at the visit and the interval between the milestone date and the visit date: age at milestone = age at visit minus the time from the milestone date to the visit date. For example, if the patient is 24 months old at the visit and the milestone occurred 6 months before the visit, the age at the milestone is 18 months.
- If no date or age for the milestone is recorded, assume the age at the milestone is the patient's age at the visit. Apply this visit-age assumption to a milestone the child has not yet met; if a milestone has clearly been achieved and only its timing is unknown, score the delay 0, since you cannot establish it was reached late.
- State in your justification which age you used and whether it came from a recorded milestone date or from the visit age.

AVOIDING OVER-SCORING:
- Do NOT score normal human variation as a phenotype. Mild or subtle physical differences are normal variants unless the clinician explicitly flags them as dysmorphic or clinically significant.
- Do NOT score common benign findings as phenotypes unless the clinician identifies the finding as clinically significant, extensive, or relevant to a diagnosis.
- Qualifiers like "mild," "slight," "subtle," or "minor" should raise your threshold — these findings should only be scored as present if the clinician explicitly connects them to a diagnosis, syndrome, or clinical concern.
- Conversely, if the clinician carries a physical observation into their diagnostic reasoning — naming it in the assessment, impression, clinical concern, or differential — score it 1, even when it was first noted only in passing during the exam. The clinician tying a finding to their diagnostic thinking is itself the signal that it is clinically significant.
- When in doubt, score 0. It is better to miss a borderline finding than to falsely attribute a phenotype based on normal variation.

FACIAL DYSMORPHISM AND EAR MALFORMATION:
- Score facial dysmorphism or ear malformation only when the finding is clinically relevant. Many facial and ear characteristics reflect normal individual variation and are not, on their own, diagnostically meaningful.
- A dysmorphism or malformation that the clinician states plainly as clinically significant, or names as a specific recognized anomaly (e.g., microtia, cleft lip or palate, coloboma), is scored 1 regardless of where in the note it appears. 
- For borderline or subtle facial or ear findings, score 1 ONLY if the clinician acknowledges the dysmorphism or malformation as part of their diagnostic reasoning — in the assessment, impression, clinical concern, or differential — rather than recording it merely as an isolated observation in the physical exam. A borderline feature noted only on physical exam, without being connected to the clinician's diagnostic thinking, should be scored 0.

NON-EXHAUSTIVE EXAMPLES OF THESE PRINCIPLES:
The following illustrate how the above rules apply in practice. These are examples only — apply the same reasoning to ALL phenotypes, not just those mentioned here.
- A "slightly recessed jaw" or "mild facial asymmetry" noted on exam does not constitute facial dysmorphism unless the clinician identifies it as dysmorphic or links it to a syndrome.
- A birthmark or single café-au-lait spot mentioned in passing does not constitute a skin pigmentation phenotype unless the clinician describes it as clinically significant, numerous, or relevant to a diagnosis.
- Height at the 15th percentile is below average but within normal range and does not qualify as short stature. Height at or below the 3rd percentile does.
- A child speaking first words at 14 months is within the normal developmental window and does not qualify as speech delay, even if slightly later than the population mean.
- Low-set or mildly prominent ears noted only under the physical exam, with no further mention, are a borderline finding and should be scored 0. If the clinician carries that finding into their diagnostic reasoning — e.g., listing "low-set, posteriorly rotated ears" among the dysmorphic features supporting a suspected syndrome in the assessment — score 1. Marked microtia or a clearly malformed pinna is scored 1 regardless of where it is documented.
- A QTc interval must be read against age-appropriate norms before it is scored as a long-QT or arrhythmia phenotype: neonates and infants normally have a longer QTc than older children, and most borderline neonatal prolongation resolves on follow-up. A single borderline QTc — roughly 450-470 ms in an infant, or 431-450 ms in an older child — should be scored 0 unless the note documents that the prolongation persisted on a repeat ECG or that long QT was diagnosed. A clearly prolonged QTc (>=500 ms at any age) or an explicit long-QT/LQTS diagnosis is scored 1 regardless of whether a history is given.
- Speech/language delay is judged against CDC milestone ages (the age by which ~75% of children show the skill): no first words besides "mama"/"dada" by 18 months, fewer than three words by 18 months, no two-word combinations (e.g., "more milk") by 24 months, or fewer than ~50 words by 30 months each meets the threshold for delay. A child saying first words at 16 months, or combining two words by 22 months, is within the normal window and does NOT qualify.
- Gross motor delay is judged against CDC/WHO milestone ages: not sitting without support by 9 months, or not walking independently ("without holding on to anyone or anything") by 18 months — the upper bound of the WHO walking window — meets the threshold. A child walking at 13-15 months is within the normal window and does NOT qualify. Absence of crawling is NOT a delay, as crawling is not a CDC milestone.
- Fine motor delay is judged against CDC milestone ages: no pincer grasp (picking up a small object between thumb and forefinger) by 12 months, or inability to hold an object in one hand while manipulating it with the other by 24 months, meets the threshold. An immature raking grasp at 9 months is age-appropriate and does NOT qualify.

PHENOTYPES:
{phenotype_list_str}

EXAMPLE OUTPUT FORMAT:
{{"Phenotypes": {{"Prematurity": {{"score": 0, "justification": "The note does not mention gestational age or premature birth.", "quote": ""}}, "Seizures": {{"score": 1, "justification": "The note documents a history of generalized tonic-clonic seizures beginning at age 3.", "quote": "Patient experienced generalized tonic-clonic seizures starting at age 3, currently managed with levetiracetam."}}, ...}}}}"""

OUTPUT_PATH = "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full.json"
SAVE_EVERY = 10
saved_count = 0


results_claude_full = []

total_input_tokens = 0
total_output_tokens = 0
saved_count = 0

def save_progress():
    pd.DataFrame(results_claude_full).to_json(OUTPUT_PATH, orient="records", indent=2)

for idx, row in notes_enrolled4.iterrows():
    subject_id = row['Subject ID']
    note_text = row['NOTE_TEXT']
    user_message = f"Clinical note for patient {subject_id}:\n\n{note_text}"
    
    max_retries = 3
    parsed = False
    
    for attempt in range(max_retries):
        with client.messages.stream(
            model="claude-sonnet-4-6",
            max_tokens=128000,
            system=system_prompt,
            messages=[{"role": "user", "content": user_message}],
            temperature=0
        ) as stream:
            response = stream.get_final_message()
        
        print(f"{subject_id} (attempt {attempt+1}) — input: {response.usage.input_tokens:,} | output: {response.usage.output_tokens:,} | stop: {response.stop_reason}")
        
        total_input_tokens += response.usage.input_tokens
        total_output_tokens += response.usage.output_tokens
        
        try:
            raw_text = response.content[0].text
            clean_text = raw_text.strip()
            if clean_text.startswith("```"):
                clean_text = clean_text.split("```")[1]
                if clean_text.startswith("json"):
                    clean_text = clean_text[4:]
                clean_text = clean_text.strip()
            
            output = json.loads(clean_text)
            phenos = output.get("Phenotypes", {})
            for pheno_name, details in phenos.items():
                results_claude_full.append({
                    "Subject ID": subject_id,
                    "Phenotype": pheno_name,
                    "Score": details.get("score", 0),
                    "Justification": details.get("justification", ""),
                    "Quote": details.get("quote", "")
                })
            parsed = True
            break
        
        except (json.JSONDecodeError, Exception) as e:
            print(f"  Attempt {attempt+1} failed for {subject_id}: {type(e).__name__}: {e}")
            if attempt < max_retries - 1:
                print(f"  Retrying in 2 seconds...")
                time.sleep(2)
            else:
                print(f"  All {max_retries} attempts failed for {subject_id}, recording PARSE ERROR")
                results_claude_full.append({"Subject ID": subject_id, "Phenotype": "PARSE ERROR"})
    
    time.sleep(1)  # small delay between notes
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} notes...")

    saved_count += 1
    if saved_count % SAVE_EVERY == 0:
        save_progress()
        print(f"Processed and saved {saved_count} notes...")

# final save after the loop
save_progress()
print(f"\nInput tokens: {total_input_tokens:,}")
print(f"Output tokens: {total_output_tokens:,}")
print(f"Total tokens: {total_input_tokens + total_output_tokens:,}")

results_claude_full_df = pd.DataFrame(results_claude_full)
print(f"Total rows: {len(results_claude_full_df)}")
print(f"Subjects: {results_claude_full_df['Subject ID'].nunique()}")

results_claude_full_df.to_json(
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full.json",
    orient="records", indent=2
)
print(f"Saved {len(results_claude_full_df)} rows")
display(results_claude_full_df.head(10))

14 hours 46 minutes

In [0]:
import pandas as pd

results_claude_full_df = pd.read_json("/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full.json", orient="records")
results_claude_full_df['Phenotype'] = results_claude_full_df['Phenotype'].str.strip()
display(results_claude_full_df)

In [0]:
subjects_incomplete = (
    results_claude_full_df.groupby('Subject ID')['Phenotype']
    .count()
    .loc[lambda x: x < 125]
    .reset_index()
    .rename(columns={'Phenotype': 'Phenotype Count'})
)

print(f"Subjects with fewer than 125 phenotypes scored: {len(subjects_incomplete)}")
display(subjects_incomplete)

#for subject_id in subjects_incomplete['Subject ID']:
 #   print(f"\nSubject ID: {subject_id}")
 #   display(results_claude_full_df[results_claude_full_df['Subject ID'] == subject_id])

In [0]:
parse_errors = results_claude_full_df[results_claude_full_df['Phenotype'] == 'PARSE ERROR']
print(f"Parse errors: {len(parse_errors)}")
print(f"Subjects with parse errors: {parse_errors['Subject ID'].tolist()}")

In [0]:
rerun = notes_enrolled4[notes_enrolled4['Subject ID'].isin(subjects_incomplete['Subject ID'])]
display(rerun)

In [0]:
# phenotype-specific justification
import json

phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)


phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)

system_prompt = f"""You are a world class board-certified medical geneticist performing a structured chart review.

TASK: Read the provided clinical note and determine whether each phenotype below is present (1) or absent (0) for the patient.

RULES:
- Output ONLY a valid JSON object and nothing else.
- The JSON must contain exactly one key: "Phenotypes".
- "Phenotypes": An object where each key is a phenotype name exactly as listed below, and each value is an object with three keys:
  - "score": 0 or 1
  - "justification": Why THIS SPECIFIC phenotype is present or absent. Do not summarize the entire note. Explain what evidence in the note directly relates to this phenotype, or state that no relevant evidence was found.
  - "quote": The exact word-for-word excerpt from the note that supports your decision for THIS SPECIFIC phenotype. Copy the text exactly as written in the note. Use "" if no relevant text exists.
- Each phenotype's justification and quote must pertain ONLY to that phenotype. Do not reuse the same justification or quote across multiple phenotypes unless the exact same sentence genuinely supports both.

SCORING CRITERIA:
- Use 1 ONLY if the note provides clear evidence that the patient HAS the phenotype as a clinically significant finding — a confirmed diagnosis, an objective measurement that meets a clinical threshold for pathology, or symptoms explicitly identified as pathological by the clinician.
- Use 0 if the note does not mention the phenotype, explicitly rules it out, there is insufficient evidence, or a reported value does not meet the accepted clinical threshold for that phenotype.
- If the note mentions a phenotype was historically present but has since resolved, still code as 1.
- Score based ONLY on findings documented for THIS patient. If the note describes features generally associated with a genotype, syndrome, or condition but does not confirm that this specific patient exhibits them, do not score those features as present.

CONTINUOUS AND QUANTITATIVE FEATURES:
- For any phenotype involving a measurable value, do not score based on being merely non-average.
- Score 1 ONLY if the value meets or exceeds the accepted clinical threshold for that phenotype based on CDC and WHO guidelines. For growth parameters, this generally means at or beyond the 3rd/97th percentile (±2 SD) for age and sex.
- If the note provides a raw value with the patient's age and sex, assess whether it crosses the clinical threshold. If the note already states a percentile, use that directly.
- If a value crosses the clinical threshold only marginally — sitting just past the cutoff or hovering at the borderline — do NOT score it 1 on the strength of a single reading. In that borderline case, score 1 only if the note documents a history of the finding: it has been observed on more than one occasion, or is described as persistent, recurrent, chronic, or longstanding. A lone borderline reading with no such documented history should be scored 0. Values that fall clearly and substantially beyond the threshold remain 1 regardless of whether a prior history is noted.
- A value that is below or above average but still within the normal range does NOT qualify as a phenotype. A developmental milestone that is slightly late but within the accepted normal window does NOT qualify as delayed.
- State the threshold you applied in your justification so your reasoning is transparent.

DETERMINING AGE AT A DEVELOPMENTAL MILESTONE:
- Before scoring any developmental delay, establish the patient's age at the milestone, then compare it to that milestone's threshold age.
- If the note records when the milestone occurred, use that. When it gives a date rather than an age, derive the age at the milestone from the age at the visit and the interval between the milestone date and the visit date: age at milestone = age at visit minus the time from the milestone date to the visit date. For example, if the patient is 24 months old at the visit and the milestone occurred 6 months before the visit, the age at the milestone is 18 months.
- If no date or age for the milestone is recorded, assume the age at the milestone is the patient's age at the visit. Apply this visit-age assumption to a milestone the child has not yet met; if a milestone has clearly been achieved and only its timing is unknown, score the delay 0, since you cannot establish it was reached late.
- State in your justification which age you used and whether it came from a recorded milestone date or from the visit age.

AVOIDING OVER-SCORING:
- Do NOT score normal human variation as a phenotype. Mild or subtle physical differences are normal variants unless the clinician explicitly flags them as dysmorphic or clinically significant.
- Do NOT score common benign findings as phenotypes unless the clinician identifies the finding as clinically significant, extensive, or relevant to a diagnosis.
- Qualifiers like "mild," "slight," "subtle," or "minor" should raise your threshold — these findings should only be scored as present if the clinician explicitly connects them to a diagnosis, syndrome, or clinical concern.
- Conversely, if the clinician carries a physical observation into their diagnostic reasoning — naming it in the assessment, impression, clinical concern, or differential — score it 1, even when it was first noted only in passing during the exam. The clinician tying a finding to their diagnostic thinking is itself the signal that it is clinically significant.
- When in doubt, score 0. It is better to miss a borderline finding than to falsely attribute a phenotype based on normal variation.

FACIAL DYSMORPHISM AND EAR MALFORMATION:
- Score facial dysmorphism or ear malformation only when the finding is clinically relevant. Many facial and ear characteristics reflect normal individual variation and are not, on their own, diagnostically meaningful.
- A dysmorphism or malformation that the clinician states plainly as clinically significant, or names as a specific recognized anomaly (e.g., microtia, cleft lip or palate, coloboma), is scored 1 regardless of where in the note it appears. 
- For borderline or subtle facial or ear findings, score 1 ONLY if the clinician acknowledges the dysmorphism or malformation as part of their diagnostic reasoning — in the assessment, impression, clinical concern, or differential — rather than recording it merely as an isolated observation in the physical exam. A borderline feature noted only on physical exam, without being connected to the clinician's diagnostic thinking, should be scored 0.

NON-EXHAUSTIVE EXAMPLES OF THESE PRINCIPLES:
The following illustrate how the above rules apply in practice. These are examples only — apply the same reasoning to ALL phenotypes, not just those mentioned here.
- A "slightly recessed jaw" or "mild facial asymmetry" noted on exam does not constitute facial dysmorphism unless the clinician identifies it as dysmorphic or links it to a syndrome.
- A birthmark or single café-au-lait spot mentioned in passing does not constitute a skin pigmentation phenotype unless the clinician describes it as clinically significant, numerous, or relevant to a diagnosis.
- Height at the 15th percentile is below average but within normal range and does not qualify as short stature. Height at or below the 3rd percentile does.
- A child speaking first words at 14 months is within the normal developmental window and does not qualify as speech delay, even if slightly later than the population mean.
- Low-set or mildly prominent ears noted only under the physical exam, with no further mention, are a borderline finding and should be scored 0. If the clinician carries that finding into their diagnostic reasoning — e.g., listing "low-set, posteriorly rotated ears" among the dysmorphic features supporting a suspected syndrome in the assessment — score 1. Marked microtia or a clearly malformed pinna is scored 1 regardless of where it is documented.
- A QTc interval must be read against age-appropriate norms before it is scored as a long-QT or arrhythmia phenotype: neonates and infants normally have a longer QTc than older children, and most borderline neonatal prolongation resolves on follow-up. A single borderline QTc — roughly 450-470 ms in an infant, or 431-450 ms in an older child — should be scored 0 unless the note documents that the prolongation persisted on a repeat ECG or that long QT was diagnosed. A clearly prolonged QTc (>=500 ms at any age) or an explicit long-QT/LQTS diagnosis is scored 1 regardless of whether a history is given.
- Speech/language delay is judged against CDC milestone ages (the age by which ~75% of children show the skill): no first words besides "mama"/"dada" by 18 months, fewer than three words by 18 months, no two-word combinations (e.g., "more milk") by 24 months, or fewer than ~50 words by 30 months each meets the threshold for delay. A child saying first words at 16 months, or combining two words by 22 months, is within the normal window and does NOT qualify.
- Gross motor delay is judged against CDC/WHO milestone ages: not sitting without support by 9 months, or not walking independently ("without holding on to anyone or anything") by 18 months — the upper bound of the WHO walking window — meets the threshold. A child walking at 13-15 months is within the normal window and does NOT qualify. Absence of crawling is NOT a delay, as crawling is not a CDC milestone.
- Fine motor delay is judged against CDC milestone ages: no pincer grasp (picking up a small object between thumb and forefinger) by 12 months, or inability to hold an object in one hand while manipulating it with the other by 24 months, meets the threshold. An immature raking grasp at 9 months is age-appropriate and does NOT qualify.

PHENOTYPES:
{phenotype_list_str}

EXAMPLE OUTPUT FORMAT:
{{"Phenotypes": {{"Prematurity": {{"score": 0, "justification": "The note does not mention gestational age or premature birth.", "quote": ""}}, "Seizures": {{"score": 1, "justification": "The note documents a history of generalized tonic-clonic seizures beginning at age 3.", "quote": "Patient experienced generalized tonic-clonic seizures starting at age 3, currently managed with levetiracetam."}}, ...}}}}"""

OUTPUT_PATH = "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun.json"
SAVE_EVERY = 10
saved_count = 0


results_claude_full_rerun = []

total_input_tokens = 0
total_output_tokens = 0
saved_count = 0

def save_progress():
    pd.DataFrame(results_claude_full_rerun).to_json(OUTPUT_PATH, orient="records", indent=2)

for idx, row in rerun.iterrows():
    subject_id = row['Subject ID']
    note_text = row['NOTE_TEXT']
    user_message = f"Clinical note for patient {subject_id}:\n\n{note_text}"
    
    max_retries = 10
    parsed = False
    
    for attempt in range(max_retries):
        with client.messages.stream(
            model="claude-sonnet-4-6",
            max_tokens=128000,
            system=system_prompt,
            messages=[{"role": "user", "content": user_message}],
            temperature=0
        ) as stream:
            response = stream.get_final_message()
        
        print(f"{subject_id} (attempt {attempt+1}) — input: {response.usage.input_tokens:,} | output: {response.usage.output_tokens:,} | stop: {response.stop_reason}")
        
        total_input_tokens += response.usage.input_tokens
        total_output_tokens += response.usage.output_tokens
        
        try:
            raw_text = response.content[0].text
            clean_text = raw_text.strip()
            if clean_text.startswith("```"):
                clean_text = clean_text.split("```")[1]
                if clean_text.startswith("json"):
                    clean_text = clean_text[4:]
                clean_text = clean_text.strip()
            
            output = json.loads(clean_text)
            phenos = output.get("Phenotypes", {})
            for pheno_name, details in phenos.items():
                results_claude_full_rerun.append({
                    "Subject ID": subject_id,
                    "Phenotype": pheno_name,
                    "Score": details.get("score", 0),
                    "Justification": details.get("justification", ""),
                    "Quote": details.get("quote", "")
                })
            parsed = True
            break
        
        except (json.JSONDecodeError, Exception) as e:
            print(f"  Attempt {attempt+1} failed for {subject_id}: {type(e).__name__}: {e}")
            if attempt < max_retries - 1:
                print(f"  Retrying in 2 seconds...")
                time.sleep(2)
            else:
                print(f"  All {max_retries} attempts failed for {subject_id}, recording PARSE ERROR")
                results_claude_full_rerun.append({"Subject ID": subject_id, "Phenotype": "PARSE ERROR"})
    
    time.sleep(1)  # small delay between notes
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} notes...")

    saved_count += 1
    if saved_count % SAVE_EVERY == 0:
        save_progress()
        print(f"Processed and saved {saved_count} notes...")

# final save after the loop
save_progress()
print(f"\nInput tokens: {total_input_tokens:,}")
print(f"Output tokens: {total_output_tokens:,}")
print(f"Total tokens: {total_input_tokens + total_output_tokens:,}")

results_claude_full_rerun_df = pd.DataFrame(results_claude_full_rerun)
print(f"Total rows: {len(results_claude_full_rerun_df)}")
print(f"Subjects: {results_claude_full_rerun_df['Subject ID'].nunique()}")

results_claude_full_rerun_df.to_json(
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun.json",
    orient="records", indent=2
)
print(f"Saved {len(results_claude_full_rerun_df)} rows")
display(results_claude_full_rerun_df.head(10))

Input tokens: 950,588
Output tokens: 777,716
Total tokens: 1,728,304
Total rows: 887
Subjects: 19
Saved 887 rows
3 hours 0 min

In [0]:
import pandas as pd

results_claude_full_rerun_df = pd.read_json("/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun.json", orient="records")
len(results_claude_full_rerun_df)

In [0]:
subjects_incomplete = (
    results_claude_full_rerun_df.groupby('Subject ID')['Phenotype']
    .count()
    .loc[lambda x: x < 125]
    .reset_index()
    .rename(columns={'Phenotype': 'Phenotype Count'})
)

print(f"Subjects with fewer than 125 phenotypes scored: {len(subjects_incomplete)}")
display(subjects_incomplete)


In [0]:
rerun2 = notes_enrolled4[notes_enrolled4['Subject ID'].isin(subjects_incomplete['Subject ID'])]
display(rerun2)

In [0]:
# phenotype-specific justification
import json

phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)


phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)

system_prompt = f"""You are a world class board-certified medical geneticist performing a structured chart review.

TASK: Read the provided clinical note and determine whether each phenotype below is present (1) or absent (0) for the patient.

RULES:
- Output ONLY a valid JSON object and nothing else.
- The JSON must contain exactly one key: "Phenotypes".
- "Phenotypes": An object where each key is a phenotype name exactly as listed below, and each value is an object with three keys:
  - "score": 0 or 1
  - "justification": Why THIS SPECIFIC phenotype is present or absent. Do not summarize the entire note. Explain what evidence in the note directly relates to this phenotype, or state that no relevant evidence was found.
  - "quote": The exact word-for-word excerpt from the note that supports your decision for THIS SPECIFIC phenotype. Copy the text exactly as written in the note. Use "" if no relevant text exists.
- Each phenotype's justification and quote must pertain ONLY to that phenotype. Do not reuse the same justification or quote across multiple phenotypes unless the exact same sentence genuinely supports both.

SCORING CRITERIA:
- Use 1 ONLY if the note provides clear evidence that the patient HAS the phenotype as a clinically significant finding — a confirmed diagnosis, an objective measurement that meets a clinical threshold for pathology, or symptoms explicitly identified as pathological by the clinician.
- Use 0 if the note does not mention the phenotype, explicitly rules it out, there is insufficient evidence, or a reported value does not meet the accepted clinical threshold for that phenotype.
- If the note mentions a phenotype was historically present but has since resolved, still code as 1.
- Score based ONLY on findings documented for THIS patient. If the note describes features generally associated with a genotype, syndrome, or condition but does not confirm that this specific patient exhibits them, do not score those features as present.

CONTINUOUS AND QUANTITATIVE FEATURES:
- For any phenotype involving a measurable value, do not score based on being merely non-average.
- Score 1 ONLY if the value meets or exceeds the accepted clinical threshold for that phenotype based on CDC and WHO guidelines. For growth parameters, this generally means at or beyond the 3rd/97th percentile (±2 SD) for age and sex.
- If the note provides a raw value with the patient's age and sex, assess whether it crosses the clinical threshold. If the note already states a percentile, use that directly.
- If a value crosses the clinical threshold only marginally — sitting just past the cutoff or hovering at the borderline — do NOT score it 1 on the strength of a single reading. In that borderline case, score 1 only if the note documents a history of the finding: it has been observed on more than one occasion, or is described as persistent, recurrent, chronic, or longstanding. A lone borderline reading with no such documented history should be scored 0. Values that fall clearly and substantially beyond the threshold remain 1 regardless of whether a prior history is noted.
- A value that is below or above average but still within the normal range does NOT qualify as a phenotype. A developmental milestone that is slightly late but within the accepted normal window does NOT qualify as delayed.
- State the threshold you applied in your justification so your reasoning is transparent.

DETERMINING AGE AT A DEVELOPMENTAL MILESTONE:
- Before scoring any developmental delay, establish the patient's age at the milestone, then compare it to that milestone's threshold age.
- If the note records when the milestone occurred, use that. When it gives a date rather than an age, derive the age at the milestone from the age at the visit and the interval between the milestone date and the visit date: age at milestone = age at visit minus the time from the milestone date to the visit date. For example, if the patient is 24 months old at the visit and the milestone occurred 6 months before the visit, the age at the milestone is 18 months.
- If no date or age for the milestone is recorded, assume the age at the milestone is the patient's age at the visit. Apply this visit-age assumption to a milestone the child has not yet met; if a milestone has clearly been achieved and only its timing is unknown, score the delay 0, since you cannot establish it was reached late.
- State in your justification which age you used and whether it came from a recorded milestone date or from the visit age.

AVOIDING OVER-SCORING:
- Do NOT score normal human variation as a phenotype. Mild or subtle physical differences are normal variants unless the clinician explicitly flags them as dysmorphic or clinically significant.
- Do NOT score common benign findings as phenotypes unless the clinician identifies the finding as clinically significant, extensive, or relevant to a diagnosis.
- Qualifiers like "mild," "slight," "subtle," or "minor" should raise your threshold — these findings should only be scored as present if the clinician explicitly connects them to a diagnosis, syndrome, or clinical concern.
- Conversely, if the clinician carries a physical observation into their diagnostic reasoning — naming it in the assessment, impression, clinical concern, or differential — score it 1, even when it was first noted only in passing during the exam. The clinician tying a finding to their diagnostic thinking is itself the signal that it is clinically significant.
- When in doubt, score 0. It is better to miss a borderline finding than to falsely attribute a phenotype based on normal variation.

FACIAL DYSMORPHISM AND EAR MALFORMATION:
- Score facial dysmorphism or ear malformation only when the finding is clinically relevant. Many facial and ear characteristics reflect normal individual variation and are not, on their own, diagnostically meaningful.
- A dysmorphism or malformation that the clinician states plainly as clinically significant, or names as a specific recognized anomaly (e.g., microtia, cleft lip or palate, coloboma), is scored 1 regardless of where in the note it appears. 
- For borderline or subtle facial or ear findings, score 1 ONLY if the clinician acknowledges the dysmorphism or malformation as part of their diagnostic reasoning — in the assessment, impression, clinical concern, or differential — rather than recording it merely as an isolated observation in the physical exam. A borderline feature noted only on physical exam, without being connected to the clinician's diagnostic thinking, should be scored 0.

NON-EXHAUSTIVE EXAMPLES OF THESE PRINCIPLES:
The following illustrate how the above rules apply in practice. These are examples only — apply the same reasoning to ALL phenotypes, not just those mentioned here.
- A "slightly recessed jaw" or "mild facial asymmetry" noted on exam does not constitute facial dysmorphism unless the clinician identifies it as dysmorphic or links it to a syndrome.
- A birthmark or single café-au-lait spot mentioned in passing does not constitute a skin pigmentation phenotype unless the clinician describes it as clinically significant, numerous, or relevant to a diagnosis.
- Height at the 15th percentile is below average but within normal range and does not qualify as short stature. Height at or below the 3rd percentile does.
- A child speaking first words at 14 months is within the normal developmental window and does not qualify as speech delay, even if slightly later than the population mean.
- Low-set or mildly prominent ears noted only under the physical exam, with no further mention, are a borderline finding and should be scored 0. If the clinician carries that finding into their diagnostic reasoning — e.g., listing "low-set, posteriorly rotated ears" among the dysmorphic features supporting a suspected syndrome in the assessment — score 1. Marked microtia or a clearly malformed pinna is scored 1 regardless of where it is documented.
- A QTc interval must be read against age-appropriate norms before it is scored as a long-QT or arrhythmia phenotype: neonates and infants normally have a longer QTc than older children, and most borderline neonatal prolongation resolves on follow-up. A single borderline QTc — roughly 450-470 ms in an infant, or 431-450 ms in an older child — should be scored 0 unless the note documents that the prolongation persisted on a repeat ECG or that long QT was diagnosed. A clearly prolonged QTc (>=500 ms at any age) or an explicit long-QT/LQTS diagnosis is scored 1 regardless of whether a history is given.
- Speech/language delay is judged against CDC milestone ages (the age by which ~75% of children show the skill): no first words besides "mama"/"dada" by 18 months, fewer than three words by 18 months, no two-word combinations (e.g., "more milk") by 24 months, or fewer than ~50 words by 30 months each meets the threshold for delay. A child saying first words at 16 months, or combining two words by 22 months, is within the normal window and does NOT qualify.
- Gross motor delay is judged against CDC/WHO milestone ages: not sitting without support by 9 months, or not walking independently ("without holding on to anyone or anything") by 18 months — the upper bound of the WHO walking window — meets the threshold. A child walking at 13-15 months is within the normal window and does NOT qualify. Absence of crawling is NOT a delay, as crawling is not a CDC milestone.
- Fine motor delay is judged against CDC milestone ages: no pincer grasp (picking up a small object between thumb and forefinger) by 12 months, or inability to hold an object in one hand while manipulating it with the other by 24 months, meets the threshold. An immature raking grasp at 9 months is age-appropriate and does NOT qualify.

PHENOTYPES:
{phenotype_list_str}

EXAMPLE OUTPUT FORMAT:
{{"Phenotypes": {{"Prematurity": {{"score": 0, "justification": "The note does not mention gestational age or premature birth.", "quote": ""}}, "Seizures": {{"score": 1, "justification": "The note documents a history of generalized tonic-clonic seizures beginning at age 3.", "quote": "Patient experienced generalized tonic-clonic seizures starting at age 3, currently managed with levetiracetam."}}, ...}}}}"""

OUTPUT_PATH = "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun2.json"
SAVE_EVERY = 10
saved_count = 0


results_claude_full_rerun2 = []

total_input_tokens = 0
total_output_tokens = 0
saved_count = 0

def save_progress():
    pd.DataFrame(results_claude_full_rerun2).to_json(OUTPUT_PATH, orient="records", indent=2)

for idx, row in rerun2.iterrows():
    subject_id = row['Subject ID']
    note_text = row['NOTE_TEXT']
    user_message = f"Clinical note for patient {subject_id}:\n\n{note_text}"
    
    max_retries = 10
    parsed = False
    
    for attempt in range(max_retries):
        with client.messages.stream(
            model="claude-sonnet-4-6",
            max_tokens=128000,
            system=system_prompt,
            messages=[{"role": "user", "content": user_message}],
            temperature=0
        ) as stream:
            response = stream.get_final_message()
        
        print(f"{subject_id} (attempt {attempt+1}) — input: {response.usage.input_tokens:,} | output: {response.usage.output_tokens:,} | stop: {response.stop_reason}")
        
        total_input_tokens += response.usage.input_tokens
        total_output_tokens += response.usage.output_tokens
        
        try:
            raw_text = response.content[0].text
            clean_text = raw_text.strip()
            if clean_text.startswith("```"):
                clean_text = clean_text.split("```")[1]
                if clean_text.startswith("json"):
                    clean_text = clean_text[4:]
                clean_text = clean_text.strip()
            
            output = json.loads(clean_text)
            phenos = output.get("Phenotypes", {})
            for pheno_name, details in phenos.items():
                results_claude_full_rerun2.append({
                    "Subject ID": subject_id,
                    "Phenotype": pheno_name,
                    "Score": details.get("score", 0),
                    "Justification": details.get("justification", ""),
                    "Quote": details.get("quote", "")
                })
            parsed = True
            break
        
        except (json.JSONDecodeError, Exception) as e:
            print(f"  Attempt {attempt+1} failed for {subject_id}: {type(e).__name__}: {e}")
            if attempt < max_retries - 1:
                print(f"  Retrying in 2 seconds...")
                time.sleep(2)
            else:
                print(f"  All {max_retries} attempts failed for {subject_id}, recording PARSE ERROR")
                results_claude_full_rerun2.append({"Subject ID": subject_id, "Phenotype": "PARSE ERROR"})
    
    time.sleep(1)  # small delay between notes
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} notes...")

    saved_count += 1
    if saved_count % SAVE_EVERY == 0:
        save_progress()
        print(f"Processed and saved {saved_count} notes...")

# final save after the loop
save_progress()
print(f"\nInput tokens: {total_input_tokens:,}")
print(f"Output tokens: {total_output_tokens:,}")
print(f"Total tokens: {total_input_tokens + total_output_tokens:,}")

results_claude_full_rerun2_df = pd.DataFrame(results_claude_full_rerun2)
print(f"Total rows: {len(results_claude_full_rerun2_df)}")
print(f"Subjects: {results_claude_full_rerun2_df['Subject ID'].nunique()}")

results_claude_full_rerun2_df.to_json(
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun2.json",
    orient="records", indent=2
)
print(f"Saved {len(results_claude_full_rerun2_df)} rows")
display(results_claude_full_rerun2_df.head(10))

Input tokens: 673,237
Output tokens: 544,132
Total tokens: 1,217,369
Total rows: 384
Subjects: 12
Saved 384 rows
2 hours 6 min

In [0]:
import pandas as pd

results_claude_full_rerun2_df = pd.read_json("/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun2.json", orient="records")
len(results_claude_full_rerun2_df)

In [0]:
subjects_incomplete = (
    results_claude_full_rerun2_df.groupby('Subject ID')['Phenotype']
    .count()
    .loc[lambda x: x < 125]
    .reset_index()
    .rename(columns={'Phenotype': 'Phenotype Count'})
)

print(f"Subjects with fewer than 125 phenotypes scored: {len(subjects_incomplete)}")
display(subjects_incomplete)


In [0]:
rerun3 = notes_enrolled4[notes_enrolled4['Subject ID'].isin(subjects_incomplete['Subject ID'])]
display(rerun3)

In [0]:
rerun3 = rerun3[~rerun3['Subject ID'].isin([2 WES IDs])]
display(rerun3)

In [0]:
# phenotype-specific justification
import json

phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)


phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)

system_prompt = f"""You are a world class board-certified medical geneticist performing a structured chart review.

TASK: Read the provided clinical note and determine whether each phenotype below is present (1) or absent (0) for the patient.

RULES:
- Output ONLY a valid JSON object and nothing else.
- The JSON must contain exactly one key: "Phenotypes".
- "Phenotypes": An object where each key is a phenotype name exactly as listed below, and each value is an object with three keys:
  - "score": 0 or 1
  - "justification": Why THIS SPECIFIC phenotype is present or absent. Do not summarize the entire note. Explain what evidence in the note directly relates to this phenotype, or state that no relevant evidence was found.
  - "quote": The exact word-for-word excerpt from the note that supports your decision for THIS SPECIFIC phenotype. Copy the text exactly as written in the note. Use "" if no relevant text exists.
- Each phenotype's justification and quote must pertain ONLY to that phenotype. Do not reuse the same justification or quote across multiple phenotypes unless the exact same sentence genuinely supports both.

SCORING CRITERIA:
- Use 1 ONLY if the note provides clear evidence that the patient HAS the phenotype as a clinically significant finding — a confirmed diagnosis, an objective measurement that meets a clinical threshold for pathology, or symptoms explicitly identified as pathological by the clinician.
- Use 0 if the note does not mention the phenotype, explicitly rules it out, there is insufficient evidence, or a reported value does not meet the accepted clinical threshold for that phenotype.
- If the note mentions a phenotype was historically present but has since resolved, still code as 1.
- Score based ONLY on findings documented for THIS patient. If the note describes features generally associated with a genotype, syndrome, or condition but does not confirm that this specific patient exhibits them, do not score those features as present.

CONTINUOUS AND QUANTITATIVE FEATURES:
- For any phenotype involving a measurable value, do not score based on being merely non-average.
- Score 1 ONLY if the value meets or exceeds the accepted clinical threshold for that phenotype based on CDC and WHO guidelines. For growth parameters, this generally means at or beyond the 3rd/97th percentile (±2 SD) for age and sex.
- If the note provides a raw value with the patient's age and sex, assess whether it crosses the clinical threshold. If the note already states a percentile, use that directly.
- If a value crosses the clinical threshold only marginally — sitting just past the cutoff or hovering at the borderline — do NOT score it 1 on the strength of a single reading. In that borderline case, score 1 only if the note documents a history of the finding: it has been observed on more than one occasion, or is described as persistent, recurrent, chronic, or longstanding. A lone borderline reading with no such documented history should be scored 0. Values that fall clearly and substantially beyond the threshold remain 1 regardless of whether a prior history is noted.
- A value that is below or above average but still within the normal range does NOT qualify as a phenotype. A developmental milestone that is slightly late but within the accepted normal window does NOT qualify as delayed.
- State the threshold you applied in your justification so your reasoning is transparent.

DETERMINING AGE AT A DEVELOPMENTAL MILESTONE:
- Before scoring any developmental delay, establish the patient's age at the milestone, then compare it to that milestone's threshold age.
- If the note records when the milestone occurred, use that. When it gives a date rather than an age, derive the age at the milestone from the age at the visit and the interval between the milestone date and the visit date: age at milestone = age at visit minus the time from the milestone date to the visit date. For example, if the patient is 24 months old at the visit and the milestone occurred 6 months before the visit, the age at the milestone is 18 months.
- If no date or age for the milestone is recorded, assume the age at the milestone is the patient's age at the visit. Apply this visit-age assumption to a milestone the child has not yet met; if a milestone has clearly been achieved and only its timing is unknown, score the delay 0, since you cannot establish it was reached late.
- State in your justification which age you used and whether it came from a recorded milestone date or from the visit age.

AVOIDING OVER-SCORING:
- Do NOT score normal human variation as a phenotype. Mild or subtle physical differences are normal variants unless the clinician explicitly flags them as dysmorphic or clinically significant.
- Do NOT score common benign findings as phenotypes unless the clinician identifies the finding as clinically significant, extensive, or relevant to a diagnosis.
- Qualifiers like "mild," "slight," "subtle," or "minor" should raise your threshold — these findings should only be scored as present if the clinician explicitly connects them to a diagnosis, syndrome, or clinical concern.
- Conversely, if the clinician carries a physical observation into their diagnostic reasoning — naming it in the assessment, impression, clinical concern, or differential — score it 1, even when it was first noted only in passing during the exam. The clinician tying a finding to their diagnostic thinking is itself the signal that it is clinically significant.
- When in doubt, score 0. It is better to miss a borderline finding than to falsely attribute a phenotype based on normal variation.

FACIAL DYSMORPHISM AND EAR MALFORMATION:
- Score facial dysmorphism or ear malformation only when the finding is clinically relevant. Many facial and ear characteristics reflect normal individual variation and are not, on their own, diagnostically meaningful.
- A dysmorphism or malformation that the clinician states plainly as clinically significant, or names as a specific recognized anomaly (e.g., microtia, cleft lip or palate, coloboma), is scored 1 regardless of where in the note it appears. 
- For borderline or subtle facial or ear findings, score 1 ONLY if the clinician acknowledges the dysmorphism or malformation as part of their diagnostic reasoning — in the assessment, impression, clinical concern, or differential — rather than recording it merely as an isolated observation in the physical exam. A borderline feature noted only on physical exam, without being connected to the clinician's diagnostic thinking, should be scored 0.

NON-EXHAUSTIVE EXAMPLES OF THESE PRINCIPLES:
The following illustrate how the above rules apply in practice. These are examples only — apply the same reasoning to ALL phenotypes, not just those mentioned here.
- A "slightly recessed jaw" or "mild facial asymmetry" noted on exam does not constitute facial dysmorphism unless the clinician identifies it as dysmorphic or links it to a syndrome.
- A birthmark or single café-au-lait spot mentioned in passing does not constitute a skin pigmentation phenotype unless the clinician describes it as clinically significant, numerous, or relevant to a diagnosis.
- Height at the 15th percentile is below average but within normal range and does not qualify as short stature. Height at or below the 3rd percentile does.
- A child speaking first words at 14 months is within the normal developmental window and does not qualify as speech delay, even if slightly later than the population mean.
- Low-set or mildly prominent ears noted only under the physical exam, with no further mention, are a borderline finding and should be scored 0. If the clinician carries that finding into their diagnostic reasoning — e.g., listing "low-set, posteriorly rotated ears" among the dysmorphic features supporting a suspected syndrome in the assessment — score 1. Marked microtia or a clearly malformed pinna is scored 1 regardless of where it is documented.
- A QTc interval must be read against age-appropriate norms before it is scored as a long-QT or arrhythmia phenotype: neonates and infants normally have a longer QTc than older children, and most borderline neonatal prolongation resolves on follow-up. A single borderline QTc — roughly 450-470 ms in an infant, or 431-450 ms in an older child — should be scored 0 unless the note documents that the prolongation persisted on a repeat ECG or that long QT was diagnosed. A clearly prolonged QTc (>=500 ms at any age) or an explicit long-QT/LQTS diagnosis is scored 1 regardless of whether a history is given.
- Speech/language delay is judged against CDC milestone ages (the age by which ~75% of children show the skill): no first words besides "mama"/"dada" by 18 months, fewer than three words by 18 months, no two-word combinations (e.g., "more milk") by 24 months, or fewer than ~50 words by 30 months each meets the threshold for delay. A child saying first words at 16 months, or combining two words by 22 months, is within the normal window and does NOT qualify.
- Gross motor delay is judged against CDC/WHO milestone ages: not sitting without support by 9 months, or not walking independently ("without holding on to anyone or anything") by 18 months — the upper bound of the WHO walking window — meets the threshold. A child walking at 13-15 months is within the normal window and does NOT qualify. Absence of crawling is NOT a delay, as crawling is not a CDC milestone.
- Fine motor delay is judged against CDC milestone ages: no pincer grasp (picking up a small object between thumb and forefinger) by 12 months, or inability to hold an object in one hand while manipulating it with the other by 24 months, meets the threshold. An immature raking grasp at 9 months is age-appropriate and does NOT qualify.

PHENOTYPES:
{phenotype_list_str}

EXAMPLE OUTPUT FORMAT:
{{"Phenotypes": {{"Prematurity": {{"score": 0, "justification": "The note does not mention gestational age or premature birth.", "quote": ""}}, "Seizures": {{"score": 1, "justification": "The note documents a history of generalized tonic-clonic seizures beginning at age 3.", "quote": "Patient experienced generalized tonic-clonic seizures starting at age 3, currently managed with levetiracetam."}}, ...}}}}"""

OUTPUT_PATH = "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun3.json"
SAVE_EVERY = 10
saved_count = 0


results_claude_full_rerun3 = []

total_input_tokens = 0
total_output_tokens = 0
saved_count = 0

def save_progress():
    pd.DataFrame(results_claude_full_rerun3).to_json(OUTPUT_PATH, orient="records", indent=2)

for idx, row in rerun3.iterrows():
    subject_id = row['Subject ID']
    note_text = row['NOTE_TEXT']
    user_message = f"Clinical note for patient {subject_id}:\n\n{note_text}"
    
    max_retries = 10
    parsed = False
    
    for attempt in range(max_retries):
        with client.messages.stream(
            model="claude-sonnet-4-6",
            max_tokens=128000,
            system=system_prompt,
            messages=[{"role": "user", "content": user_message}],
            temperature=0
        ) as stream:
            response = stream.get_final_message()
        
        print(f"{subject_id} (attempt {attempt+1}) — input: {response.usage.input_tokens:,} | output: {response.usage.output_tokens:,} | stop: {response.stop_reason}")
        
        total_input_tokens += response.usage.input_tokens
        total_output_tokens += response.usage.output_tokens
        
        try:
            raw_text = response.content[0].text
            clean_text = raw_text.strip()
            if clean_text.startswith("```"):
                clean_text = clean_text.split("```")[1]
                if clean_text.startswith("json"):
                    clean_text = clean_text[4:]
                clean_text = clean_text.strip()
            
            output = json.loads(clean_text)
            phenos = output.get("Phenotypes", {})
            for pheno_name, details in phenos.items():
                results_claude_full_rerun3.append({
                    "Subject ID": subject_id,
                    "Phenotype": pheno_name,
                    "Score": details.get("score", 0),
                    "Justification": details.get("justification", ""),
                    "Quote": details.get("quote", "")
                })
            parsed = True
            break
        
        except (json.JSONDecodeError, Exception) as e:
            print(f"  Attempt {attempt+1} failed for {subject_id}: {type(e).__name__}: {e}")
            if attempt < max_retries - 1:
                print(f"  Retrying in 2 seconds...")
                time.sleep(2)
            else:
                print(f"  All {max_retries} attempts failed for {subject_id}, recording PARSE ERROR")
                results_claude_full_rerun3.append({"Subject ID": subject_id, "Phenotype": "PARSE ERROR"})
    
    time.sleep(1)  # small delay between notes
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} notes...")

    saved_count += 1
    if saved_count % SAVE_EVERY == 0:
        save_progress()
        print(f"Processed and saved {saved_count} notes...")

# final save after the loop
save_progress()
print(f"\nInput tokens: {total_input_tokens:,}")
print(f"Output tokens: {total_output_tokens:,}")
print(f"Total tokens: {total_input_tokens + total_output_tokens:,}")

results_claude_full_rerun3_df = pd.DataFrame(results_claude_full_rerun3)
print(f"Total rows: {len(results_claude_full_rerun3_df)}")
print(f"Subjects: {results_claude_full_rerun3_df['Subject ID'].nunique()}")

results_claude_full_rerun3_df.to_json(
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun3.json",
    orient="records", indent=2
)
print(f"Saved {len(results_claude_full_rerun3_df)} rows")
display(results_claude_full_rerun3_df.head(10))

Input tokens: 504,876
Output tokens: 407,699
Total tokens: 912,575
Total rows: 131
Subjects: 7
Saved 131 rows
1 hour, 37 minutes

In [0]:
import pandas as pd

results_claude_full_rerun3_df = pd.read_json("/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun3.json", orient="records")
len(results_claude_full_rerun3_df)

In [0]:
subjects_incomplete = (
    results_claude_full_rerun3_df.groupby('Subject ID')['Phenotype']
    .count()
    .loc[lambda x: x < 125]
    .reset_index()
    .rename(columns={'Phenotype': 'Phenotype Count'})
)

print(f"Subjects with fewer than 125 phenotypes scored: {len(subjects_incomplete)}")
display(subjects_incomplete)


In [0]:
rerun4 = notes_enrolled4[notes_enrolled4['Subject ID'].isin(subjects_incomplete['Subject ID'])]
display(rerun4)

In [0]:
# phenotype-specific justification
import json

phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)


phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)

system_prompt = f"""You are a world class board-certified medical geneticist performing a structured chart review.

TASK: Read the provided clinical note and determine whether each phenotype below is present (1) or absent (0) for the patient.

RULES:
- Output ONLY a valid JSON object and nothing else.
- The JSON must contain exactly one key: "Phenotypes".
- "Phenotypes": An object where each key is a phenotype name exactly as listed below, and each value is an object with three keys:
  - "score": 0 or 1
  - "justification": Why THIS SPECIFIC phenotype is present or absent. Do not summarize the entire note. Explain what evidence in the note directly relates to this phenotype, or state that no relevant evidence was found.
  - "quote": The exact word-for-word excerpt from the note that supports your decision for THIS SPECIFIC phenotype. Copy the text exactly as written in the note. Use "" if no relevant text exists.
- Each phenotype's justification and quote must pertain ONLY to that phenotype. Do not reuse the same justification or quote across multiple phenotypes unless the exact same sentence genuinely supports both.

SCORING CRITERIA:
- Use 1 ONLY if the note provides clear evidence that the patient HAS the phenotype as a clinically significant finding — a confirmed diagnosis, an objective measurement that meets a clinical threshold for pathology, or symptoms explicitly identified as pathological by the clinician.
- Use 0 if the note does not mention the phenotype, explicitly rules it out, there is insufficient evidence, or a reported value does not meet the accepted clinical threshold for that phenotype.
- If the note mentions a phenotype was historically present but has since resolved, still code as 1.
- Score based ONLY on findings documented for THIS patient. If the note describes features generally associated with a genotype, syndrome, or condition but does not confirm that this specific patient exhibits them, do not score those features as present.

CONTINUOUS AND QUANTITATIVE FEATURES:
- For any phenotype involving a measurable value, do not score based on being merely non-average.
- Score 1 ONLY if the value meets or exceeds the accepted clinical threshold for that phenotype based on CDC and WHO guidelines. For growth parameters, this generally means at or beyond the 3rd/97th percentile (±2 SD) for age and sex.
- If the note provides a raw value with the patient's age and sex, assess whether it crosses the clinical threshold. If the note already states a percentile, use that directly.
- If a value crosses the clinical threshold only marginally — sitting just past the cutoff or hovering at the borderline — do NOT score it 1 on the strength of a single reading. In that borderline case, score 1 only if the note documents a history of the finding: it has been observed on more than one occasion, or is described as persistent, recurrent, chronic, or longstanding. A lone borderline reading with no such documented history should be scored 0. Values that fall clearly and substantially beyond the threshold remain 1 regardless of whether a prior history is noted.
- A value that is below or above average but still within the normal range does NOT qualify as a phenotype. A developmental milestone that is slightly late but within the accepted normal window does NOT qualify as delayed.
- State the threshold you applied in your justification so your reasoning is transparent.

DETERMINING AGE AT A DEVELOPMENTAL MILESTONE:
- Before scoring any developmental delay, establish the patient's age at the milestone, then compare it to that milestone's threshold age.
- If the note records when the milestone occurred, use that. When it gives a date rather than an age, derive the age at the milestone from the age at the visit and the interval between the milestone date and the visit date: age at milestone = age at visit minus the time from the milestone date to the visit date. For example, if the patient is 24 months old at the visit and the milestone occurred 6 months before the visit, the age at the milestone is 18 months.
- If no date or age for the milestone is recorded, assume the age at the milestone is the patient's age at the visit. Apply this visit-age assumption to a milestone the child has not yet met; if a milestone has clearly been achieved and only its timing is unknown, score the delay 0, since you cannot establish it was reached late.
- State in your justification which age you used and whether it came from a recorded milestone date or from the visit age.

AVOIDING OVER-SCORING:
- Do NOT score normal human variation as a phenotype. Mild or subtle physical differences are normal variants unless the clinician explicitly flags them as dysmorphic or clinically significant.
- Do NOT score common benign findings as phenotypes unless the clinician identifies the finding as clinically significant, extensive, or relevant to a diagnosis.
- Qualifiers like "mild," "slight," "subtle," or "minor" should raise your threshold — these findings should only be scored as present if the clinician explicitly connects them to a diagnosis, syndrome, or clinical concern.
- Conversely, if the clinician carries a physical observation into their diagnostic reasoning — naming it in the assessment, impression, clinical concern, or differential — score it 1, even when it was first noted only in passing during the exam. The clinician tying a finding to their diagnostic thinking is itself the signal that it is clinically significant.
- When in doubt, score 0. It is better to miss a borderline finding than to falsely attribute a phenotype based on normal variation.

FACIAL DYSMORPHISM AND EAR MALFORMATION:
- Score facial dysmorphism or ear malformation only when the finding is clinically relevant. Many facial and ear characteristics reflect normal individual variation and are not, on their own, diagnostically meaningful.
- A dysmorphism or malformation that the clinician states plainly as clinically significant, or names as a specific recognized anomaly (e.g., microtia, cleft lip or palate, coloboma), is scored 1 regardless of where in the note it appears. 
- For borderline or subtle facial or ear findings, score 1 ONLY if the clinician acknowledges the dysmorphism or malformation as part of their diagnostic reasoning — in the assessment, impression, clinical concern, or differential — rather than recording it merely as an isolated observation in the physical exam. A borderline feature noted only on physical exam, without being connected to the clinician's diagnostic thinking, should be scored 0.

NON-EXHAUSTIVE EXAMPLES OF THESE PRINCIPLES:
The following illustrate how the above rules apply in practice. These are examples only — apply the same reasoning to ALL phenotypes, not just those mentioned here.
- A "slightly recessed jaw" or "mild facial asymmetry" noted on exam does not constitute facial dysmorphism unless the clinician identifies it as dysmorphic or links it to a syndrome.
- A birthmark or single café-au-lait spot mentioned in passing does not constitute a skin pigmentation phenotype unless the clinician describes it as clinically significant, numerous, or relevant to a diagnosis.
- Height at the 15th percentile is below average but within normal range and does not qualify as short stature. Height at or below the 3rd percentile does.
- A child speaking first words at 14 months is within the normal developmental window and does not qualify as speech delay, even if slightly later than the population mean.
- Low-set or mildly prominent ears noted only under the physical exam, with no further mention, are a borderline finding and should be scored 0. If the clinician carries that finding into their diagnostic reasoning — e.g., listing "low-set, posteriorly rotated ears" among the dysmorphic features supporting a suspected syndrome in the assessment — score 1. Marked microtia or a clearly malformed pinna is scored 1 regardless of where it is documented.
- A QTc interval must be read against age-appropriate norms before it is scored as a long-QT or arrhythmia phenotype: neonates and infants normally have a longer QTc than older children, and most borderline neonatal prolongation resolves on follow-up. A single borderline QTc — roughly 450-470 ms in an infant, or 431-450 ms in an older child — should be scored 0 unless the note documents that the prolongation persisted on a repeat ECG or that long QT was diagnosed. A clearly prolonged QTc (>=500 ms at any age) or an explicit long-QT/LQTS diagnosis is scored 1 regardless of whether a history is given.
- Speech/language delay is judged against CDC milestone ages (the age by which ~75% of children show the skill): no first words besides "mama"/"dada" by 18 months, fewer than three words by 18 months, no two-word combinations (e.g., "more milk") by 24 months, or fewer than ~50 words by 30 months each meets the threshold for delay. A child saying first words at 16 months, or combining two words by 22 months, is within the normal window and does NOT qualify.
- Gross motor delay is judged against CDC/WHO milestone ages: not sitting without support by 9 months, or not walking independently ("without holding on to anyone or anything") by 18 months — the upper bound of the WHO walking window — meets the threshold. A child walking at 13-15 months is within the normal window and does NOT qualify. Absence of crawling is NOT a delay, as crawling is not a CDC milestone.
- Fine motor delay is judged against CDC milestone ages: no pincer grasp (picking up a small object between thumb and forefinger) by 12 months, or inability to hold an object in one hand while manipulating it with the other by 24 months, meets the threshold. An immature raking grasp at 9 months is age-appropriate and does NOT qualify.

PHENOTYPES:
{phenotype_list_str}

EXAMPLE OUTPUT FORMAT:
{{"Phenotypes": {{"Prematurity": {{"score": 0, "justification": "The note does not mention gestational age or premature birth.", "quote": ""}}, "Seizures": {{"score": 1, "justification": "The note documents a history of generalized tonic-clonic seizures beginning at age 3.", "quote": "Patient experienced generalized tonic-clonic seizures starting at age 3, currently managed with levetiracetam."}}, ...}}}}"""

OUTPUT_PATH = "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun4.json"
SAVE_EVERY = 10
saved_count = 0


results_claude_full_rerun4 = []

total_input_tokens = 0
total_output_tokens = 0
saved_count = 0

def save_progress():
    pd.DataFrame(results_claude_full_rerun4).to_json(OUTPUT_PATH, orient="records", indent=2)

for idx, row in rerun4.iterrows():
    subject_id = row['Subject ID']
    note_text = row['NOTE_TEXT']
    user_message = f"Clinical note for patient {subject_id}:\n\n{note_text}"
    
    max_retries = 4
    parsed = False
    
    for attempt in range(max_retries):
        with client.messages.stream(
            model="claude-sonnet-4-6",
            max_tokens=128000,
            system=system_prompt,
            messages=[{"role": "user", "content": user_message}],
            temperature=0
        ) as stream:
            response = stream.get_final_message()
        
        print(f"{subject_id} (attempt {attempt+1}) — input: {response.usage.input_tokens:,} | output: {response.usage.output_tokens:,} | stop: {response.stop_reason}")
        
        total_input_tokens += response.usage.input_tokens
        total_output_tokens += response.usage.output_tokens
        
        try:
            raw_text = response.content[0].text
            clean_text = raw_text.strip()
            if clean_text.startswith("```"):
                clean_text = clean_text.split("```")[1]
                if clean_text.startswith("json"):
                    clean_text = clean_text[4:]
                clean_text = clean_text.strip()
            
            output = json.loads(clean_text)
            phenos = output.get("Phenotypes", {})
            for pheno_name, details in phenos.items():
                results_claude_full_rerun4.append({
                    "Subject ID": subject_id,
                    "Phenotype": pheno_name,
                    "Score": details.get("score", 0),
                    "Justification": details.get("justification", ""),
                    "Quote": details.get("quote", "")
                })
            parsed = True
            break
        
        except (json.JSONDecodeError, Exception) as e:
            print(f"  Attempt {attempt+1} failed for {subject_id}: {type(e).__name__}: {e}")
            if attempt < max_retries - 1:
                print(f"  Retrying in 2 seconds...")
                time.sleep(2)
            else:
                print(f"  All {max_retries} attempts failed for {subject_id}, recording PARSE ERROR")
                results_claude_full_rerun4.append({"Subject ID": subject_id, "Phenotype": "PARSE ERROR"})
    
    time.sleep(1)  # small delay between notes
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} notes...")

    saved_count += 1
    if saved_count % SAVE_EVERY == 0:
        save_progress()
        print(f"Processed and saved {saved_count} notes...")

# final save after the loop
save_progress()
print(f"\nInput tokens: {total_input_tokens:,}")
print(f"Output tokens: {total_output_tokens:,}")
print(f"Total tokens: {total_input_tokens + total_output_tokens:,}")

results_claude_full_rerun4_df = pd.DataFrame(results_claude_full_rerun4)
print(f"Total rows: {len(results_claude_full_rerun4_df)}")
print(f"Subjects: {results_claude_full_rerun4_df['Subject ID'].nunique()}")

results_claude_full_rerun4_df.to_json(
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun4.json",
    orient="records", indent=2
)
print(f"Saved {len(results_claude_full_rerun4_df)} rows")
display(results_claude_full_rerun4_df.head(10))

Input tokens: 199,008
Output tokens: 158,533
Total tokens: 357,541
Total rows: 6
Subjects: 6
Saved 6 rows
37 minutes
All parse errors

In [0]:
import pandas as pd

results_claude_full_rerun4_df = pd.read_json("/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun4.json", orient="records")
len(results_claude_full_rerun4_df)

In [0]:
subjects_incomplete = (
    results_claude_full_rerun4_df.groupby('Subject ID')['Phenotype']
    .count()
    .loc[lambda x: x < 125]
    .reset_index()
    .rename(columns={'Phenotype': 'Phenotype Count'})
)

print(f"Subjects with fewer than 125 phenotypes scored: {len(subjects_incomplete)}")
display(subjects_incomplete)

In [0]:
rerun5 = notes_enrolled4[notes_enrolled4['Subject ID'].isin(subjects_incomplete['Subject ID'])].copy()
rerun5['NOTE_TEXT'] = rerun5['NOTE_TEXT'].str.strip()
display(rerun5)

In [0]:
# phenotype-specific justification
import json

phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)


phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)

system_prompt = f"""You are a world class board-certified medical geneticist performing a structured chart review.

TASK: Read the provided clinical note and determine whether each phenotype below is present (1) or absent (0) for the patient.

RULES:
- Output ONLY a valid JSON object and nothing else.
- The JSON must contain exactly one key: "Phenotypes".
- "Phenotypes": An object where each key is a phenotype name exactly as listed below, and each value is an object with three keys:
  - "score": 0 or 1
  - "justification": Why THIS SPECIFIC phenotype is present or absent. Do not summarize the entire note. Explain what evidence in the note directly relates to this phenotype, or state that no relevant evidence was found.
  - "quote": The exact word-for-word excerpt from the note that supports your decision for THIS SPECIFIC phenotype. Copy the text exactly as written in the note. Use "" if no relevant text exists.
- Each phenotype's justification and quote must pertain ONLY to that phenotype. Do not reuse the same justification or quote across multiple phenotypes unless the exact same sentence genuinely supports both.

SCORING CRITERIA:
- Use 1 ONLY if the note provides clear evidence that the patient HAS the phenotype as a clinically significant finding — a confirmed diagnosis, an objective measurement that meets a clinical threshold for pathology, or symptoms explicitly identified as pathological by the clinician.
- Use 0 if the note does not mention the phenotype, explicitly rules it out, there is insufficient evidence, or a reported value does not meet the accepted clinical threshold for that phenotype.
- If the note mentions a phenotype was historically present but has since resolved, still code as 1.
- Score based ONLY on findings documented for THIS patient. If the note describes features generally associated with a genotype, syndrome, or condition but does not confirm that this specific patient exhibits them, do not score those features as present.

CONTINUOUS AND QUANTITATIVE FEATURES:
- For any phenotype involving a measurable value, do not score based on being merely non-average.
- Score 1 ONLY if the value meets or exceeds the accepted clinical threshold for that phenotype based on CDC and WHO guidelines. For growth parameters, this generally means at or beyond the 3rd/97th percentile (±2 SD) for age and sex.
- If the note provides a raw value with the patient's age and sex, assess whether it crosses the clinical threshold. If the note already states a percentile, use that directly.
- If a value crosses the clinical threshold only marginally — sitting just past the cutoff or hovering at the borderline — do NOT score it 1 on the strength of a single reading. In that borderline case, score 1 only if the note documents a history of the finding: it has been observed on more than one occasion, or is described as persistent, recurrent, chronic, or longstanding. A lone borderline reading with no such documented history should be scored 0. Values that fall clearly and substantially beyond the threshold remain 1 regardless of whether a prior history is noted.
- A value that is below or above average but still within the normal range does NOT qualify as a phenotype. A developmental milestone that is slightly late but within the accepted normal window does NOT qualify as delayed.
- State the threshold you applied in your justification so your reasoning is transparent.

DETERMINING AGE AT A DEVELOPMENTAL MILESTONE:
- Before scoring any developmental delay, establish the patient's age at the milestone, then compare it to that milestone's threshold age.
- If the note records when the milestone occurred, use that. When it gives a date rather than an age, derive the age at the milestone from the age at the visit and the interval between the milestone date and the visit date: age at milestone = age at visit minus the time from the milestone date to the visit date. For example, if the patient is 24 months old at the visit and the milestone occurred 6 months before the visit, the age at the milestone is 18 months.
- If no date or age for the milestone is recorded, assume the age at the milestone is the patient's age at the visit. Apply this visit-age assumption to a milestone the child has not yet met; if a milestone has clearly been achieved and only its timing is unknown, score the delay 0, since you cannot establish it was reached late.
- State in your justification which age you used and whether it came from a recorded milestone date or from the visit age.

AVOIDING OVER-SCORING:
- Do NOT score normal human variation as a phenotype. Mild or subtle physical differences are normal variants unless the clinician explicitly flags them as dysmorphic or clinically significant.
- Do NOT score common benign findings as phenotypes unless the clinician identifies the finding as clinically significant, extensive, or relevant to a diagnosis.
- Qualifiers like "mild," "slight," "subtle," or "minor" should raise your threshold — these findings should only be scored as present if the clinician explicitly connects them to a diagnosis, syndrome, or clinical concern.
- Conversely, if the clinician carries a physical observation into their diagnostic reasoning — naming it in the assessment, impression, clinical concern, or differential — score it 1, even when it was first noted only in passing during the exam. The clinician tying a finding to their diagnostic thinking is itself the signal that it is clinically significant.
- When in doubt, score 0. It is better to miss a borderline finding than to falsely attribute a phenotype based on normal variation.

FACIAL DYSMORPHISM AND EAR MALFORMATION:
- Score facial dysmorphism or ear malformation only when the finding is clinically relevant. Many facial and ear characteristics reflect normal individual variation and are not, on their own, diagnostically meaningful.
- A dysmorphism or malformation that the clinician states plainly as clinically significant, or names as a specific recognized anomaly (e.g., microtia, cleft lip or palate, coloboma), is scored 1 regardless of where in the note it appears. 
- For borderline or subtle facial or ear findings, score 1 ONLY if the clinician acknowledges the dysmorphism or malformation as part of their diagnostic reasoning — in the assessment, impression, clinical concern, or differential — rather than recording it merely as an isolated observation in the physical exam. A borderline feature noted only on physical exam, without being connected to the clinician's diagnostic thinking, should be scored 0.

NON-EXHAUSTIVE EXAMPLES OF THESE PRINCIPLES:
The following illustrate how the above rules apply in practice. These are examples only — apply the same reasoning to ALL phenotypes, not just those mentioned here.
- A "slightly recessed jaw" or "mild facial asymmetry" noted on exam does not constitute facial dysmorphism unless the clinician identifies it as dysmorphic or links it to a syndrome.
- A birthmark or single café-au-lait spot mentioned in passing does not constitute a skin pigmentation phenotype unless the clinician describes it as clinically significant, numerous, or relevant to a diagnosis.
- Height at the 15th percentile is below average but within normal range and does not qualify as short stature. Height at or below the 3rd percentile does.
- A child speaking first words at 14 months is within the normal developmental window and does not qualify as speech delay, even if slightly later than the population mean.
- Low-set or mildly prominent ears noted only under the physical exam, with no further mention, are a borderline finding and should be scored 0. If the clinician carries that finding into their diagnostic reasoning — e.g., listing "low-set, posteriorly rotated ears" among the dysmorphic features supporting a suspected syndrome in the assessment — score 1. Marked microtia or a clearly malformed pinna is scored 1 regardless of where it is documented.
- A QTc interval must be read against age-appropriate norms before it is scored as a long-QT or arrhythmia phenotype: neonates and infants normally have a longer QTc than older children, and most borderline neonatal prolongation resolves on follow-up. A single borderline QTc — roughly 450-470 ms in an infant, or 431-450 ms in an older child — should be scored 0 unless the note documents that the prolongation persisted on a repeat ECG or that long QT was diagnosed. A clearly prolonged QTc (>=500 ms at any age) or an explicit long-QT/LQTS diagnosis is scored 1 regardless of whether a history is given.
- Speech/language delay is judged against CDC milestone ages (the age by which ~75% of children show the skill): no first words besides "mama"/"dada" by 18 months, fewer than three words by 18 months, no two-word combinations (e.g., "more milk") by 24 months, or fewer than ~50 words by 30 months each meets the threshold for delay. A child saying first words at 16 months, or combining two words by 22 months, is within the normal window and does NOT qualify.
- Gross motor delay is judged against CDC/WHO milestone ages: not sitting without support by 9 months, or not walking independently ("without holding on to anyone or anything") by 18 months — the upper bound of the WHO walking window — meets the threshold. A child walking at 13-15 months is within the normal window and does NOT qualify. Absence of crawling is NOT a delay, as crawling is not a CDC milestone.
- Fine motor delay is judged against CDC milestone ages: no pincer grasp (picking up a small object between thumb and forefinger) by 12 months, or inability to hold an object in one hand while manipulating it with the other by 24 months, meets the threshold. An immature raking grasp at 9 months is age-appropriate and does NOT qualify.

PHENOTYPES:
{phenotype_list_str}

EXAMPLE OUTPUT FORMAT:
{{"Phenotypes": {{"Prematurity": {{"score": 0, "justification": "The note does not mention gestational age or premature birth.", "quote": ""}}, "Seizures": {{"score": 1, "justification": "The note documents a history of generalized tonic-clonic seizures beginning at age 3.", "quote": "Patient experienced generalized tonic-clonic seizures starting at age 3, currently managed with levetiracetam."}}, ...}}}}"""

OUTPUT_PATH = "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun5.json"
SAVE_EVERY = 10
saved_count = 0


results_claude_full_rerun5 = []

total_input_tokens = 0
total_output_tokens = 0
saved_count = 0

def save_progress():
    pd.DataFrame(results_claude_full_rerun5).to_json(OUTPUT_PATH, orient="records", indent=2)

for idx, row in rerun5.iterrows():
    subject_id = row['Subject ID']
    note_text = row['NOTE_TEXT']
    user_message = f"Clinical note for patient {subject_id}:\n\n{note_text}"
    
    max_retries = 4
    parsed = False
    
    for attempt in range(max_retries):
        with client.messages.stream(
            model="claude-sonnet-4-6",
            max_tokens=128000,
            system=system_prompt,
            messages=[{"role": "user", "content": user_message}],
            temperature=0
        ) as stream:
            response = stream.get_final_message()
        
        print(f"{subject_id} (attempt {attempt+1}) — input: {response.usage.input_tokens:,} | output: {response.usage.output_tokens:,} | stop: {response.stop_reason}")
        
        total_input_tokens += response.usage.input_tokens
        total_output_tokens += response.usage.output_tokens
        
        try:
            raw_text = response.content[0].text
            clean_text = raw_text.strip()
            if clean_text.startswith("```"):
                clean_text = clean_text.split("```")[1]
                if clean_text.startswith("json"):
                    clean_text = clean_text[4:]
                clean_text = clean_text.strip()
            
            output = json.loads(clean_text)
            phenos = output.get("Phenotypes", {})
            for pheno_name, details in phenos.items():
                results_claude_full_rerun5.append({
                    "Subject ID": subject_id,
                    "Phenotype": pheno_name,
                    "Score": details.get("score", 0),
                    "Justification": details.get("justification", ""),
                    "Quote": details.get("quote", "")
                })
            parsed = True
            break
        
        except (json.JSONDecodeError, Exception) as e:
            print(f"  Attempt {attempt+1} failed for {subject_id}: {type(e).__name__}: {e}")
            if attempt < max_retries - 1:
                print(f"  Retrying in 2 seconds...")
                time.sleep(2)
            else:
                print(f"  All {max_retries} attempts failed for {subject_id}, recording PARSE ERROR")
                results_claude_full_rerun5.append({"Subject ID": subject_id, "Phenotype": "PARSE ERROR"})
    
    time.sleep(1)  # small delay between notes
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} notes...")

    saved_count += 1
    if saved_count % SAVE_EVERY == 0:
        save_progress()
        print(f"Processed and saved {saved_count} notes...")

# final save after the loop
save_progress()
print(f"\nInput tokens: {total_input_tokens:,}")
print(f"Output tokens: {total_output_tokens:,}")
print(f"Total tokens: {total_input_tokens + total_output_tokens:,}")

results_claude_full_rerun5_df = pd.DataFrame(results_claude_full_rerun5)
print(f"Total rows: {len(results_claude_full_rerun5_df)}")
print(f"Subjects: {results_claude_full_rerun5_df['Subject ID'].nunique()}")

results_claude_full_rerun5_df.to_json(
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun5.json",
    orient="records", indent=2
)
print(f"Saved {len(results_claude_full_rerun5_df)} rows")
display(results_claude_full_rerun5_df.head(10))

Input tokens: 172,564
Output tokens: 142,002
Total tokens: 314,566
Total rows: 130
Subjects: 6
Saved 130 rows

In [0]:
results_claude_full_rerun5_df = pd.read_json("/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun5.json", orient="records")
display(results_claude_full_rerun5_df)

In [0]:
subjects_incomplete = (
    results_claude_full_rerun5_df.groupby('Subject ID')['Phenotype']
    .count()
    .loc[lambda x: x < 125]
    .reset_index()
    .rename(columns={'Phenotype': 'Phenotype Count'})
)

print(f"Subjects with fewer than 125 phenotypes scored: {len(subjects_incomplete)}")
display(subjects_incomplete)

#for subject_id in subjects_incomplete['Subject ID']:
 #   print(f"\nSubject ID: {subject_id}")
 #   display(results_claude_full_rerun5_df[results_claude_full_rerun5_df['Subject ID'] == subject_id])

In [0]:
rerun6 = notes_enrolled4[notes_enrolled4['Subject ID'].isin(subjects_incomplete['Subject ID'])].copy()
display(rerun6)

In [0]:
# phenotype-specific justification
import json

phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)


phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)

system_prompt = f"""You are a world class board-certified medical geneticist performing a structured chart review.

TASK: Read the provided clinical note and determine whether each phenotype below is present (1) or absent (0) for the patient.

RULES:
- Output ONLY a valid JSON object and nothing else.
- The JSON must contain exactly one key: "Phenotypes".
- "Phenotypes": An object where each key is a phenotype name exactly as listed below, and each value is an object with three keys:
  - "score": 0 or 1
  - "justification": Why THIS SPECIFIC phenotype is present or absent. Do not summarize the entire note. Explain what evidence in the note directly relates to this phenotype, or state that no relevant evidence was found.
  - "quote": The exact word-for-word excerpt from the note that supports your decision for THIS SPECIFIC phenotype. Copy the text exactly as written in the note. Use "" if no relevant text exists.
- Each phenotype's justification and quote must pertain ONLY to that phenotype. Do not reuse the same justification or quote across multiple phenotypes unless the exact same sentence genuinely supports both.

SCORING CRITERIA:
- Use 1 ONLY if the note provides clear evidence that the patient HAS the phenotype as a clinically significant finding — a confirmed diagnosis, an objective measurement that meets a clinical threshold for pathology, or symptoms explicitly identified as pathological by the clinician.
- Use 0 if the note does not mention the phenotype, explicitly rules it out, there is insufficient evidence, or a reported value does not meet the accepted clinical threshold for that phenotype.
- If the note mentions a phenotype was historically present but has since resolved, still code as 1.
- Score based ONLY on findings documented for THIS patient. If the note describes features generally associated with a genotype, syndrome, or condition but does not confirm that this specific patient exhibits them, do not score those features as present.

CONTINUOUS AND QUANTITATIVE FEATURES:
- For any phenotype involving a measurable value, do not score based on being merely non-average.
- Score 1 ONLY if the value meets or exceeds the accepted clinical threshold for that phenotype based on CDC and WHO guidelines. For growth parameters, this generally means at or beyond the 3rd/97th percentile (±2 SD) for age and sex.
- If the note provides a raw value with the patient's age and sex, assess whether it crosses the clinical threshold. If the note already states a percentile, use that directly.
- If a value crosses the clinical threshold only marginally — sitting just past the cutoff or hovering at the borderline — do NOT score it 1 on the strength of a single reading. In that borderline case, score 1 only if the note documents a history of the finding: it has been observed on more than one occasion, or is described as persistent, recurrent, chronic, or longstanding. A lone borderline reading with no such documented history should be scored 0. Values that fall clearly and substantially beyond the threshold remain 1 regardless of whether a prior history is noted.
- A value that is below or above average but still within the normal range does NOT qualify as a phenotype. A developmental milestone that is slightly late but within the accepted normal window does NOT qualify as delayed.
- State the threshold you applied in your justification so your reasoning is transparent.

DETERMINING AGE AT A DEVELOPMENTAL MILESTONE:
- Before scoring any developmental delay, establish the patient's age at the milestone, then compare it to that milestone's threshold age.
- If the note records when the milestone occurred, use that. When it gives a date rather than an age, derive the age at the milestone from the age at the visit and the interval between the milestone date and the visit date: age at milestone = age at visit minus the time from the milestone date to the visit date. For example, if the patient is 24 months old at the visit and the milestone occurred 6 months before the visit, the age at the milestone is 18 months.
- If no date or age for the milestone is recorded, assume the age at the milestone is the patient's age at the visit. Apply this visit-age assumption to a milestone the child has not yet met; if a milestone has clearly been achieved and only its timing is unknown, score the delay 0, since you cannot establish it was reached late.
- State in your justification which age you used and whether it came from a recorded milestone date or from the visit age.

AVOIDING OVER-SCORING:
- Do NOT score normal human variation as a phenotype. Mild or subtle physical differences are normal variants unless the clinician explicitly flags them as dysmorphic or clinically significant.
- Do NOT score common benign findings as phenotypes unless the clinician identifies the finding as clinically significant, extensive, or relevant to a diagnosis.
- Qualifiers like "mild," "slight," "subtle," or "minor" should raise your threshold — these findings should only be scored as present if the clinician explicitly connects them to a diagnosis, syndrome, or clinical concern.
- Conversely, if the clinician carries a physical observation into their diagnostic reasoning — naming it in the assessment, impression, clinical concern, or differential — score it 1, even when it was first noted only in passing during the exam. The clinician tying a finding to their diagnostic thinking is itself the signal that it is clinically significant.
- When in doubt, score 0. It is better to miss a borderline finding than to falsely attribute a phenotype based on normal variation.

FACIAL DYSMORPHISM AND EAR MALFORMATION:
- Score facial dysmorphism or ear malformation only when the finding is clinically relevant. Many facial and ear characteristics reflect normal individual variation and are not, on their own, diagnostically meaningful.
- A dysmorphism or malformation that the clinician states plainly as clinically significant, or names as a specific recognized anomaly (e.g., microtia, cleft lip or palate, coloboma), is scored 1 regardless of where in the note it appears. 
- For borderline or subtle facial or ear findings, score 1 ONLY if the clinician acknowledges the dysmorphism or malformation as part of their diagnostic reasoning — in the assessment, impression, clinical concern, or differential — rather than recording it merely as an isolated observation in the physical exam. A borderline feature noted only on physical exam, without being connected to the clinician's diagnostic thinking, should be scored 0.

NON-EXHAUSTIVE EXAMPLES OF THESE PRINCIPLES:
The following illustrate how the above rules apply in practice. These are examples only — apply the same reasoning to ALL phenotypes, not just those mentioned here.
- A "slightly recessed jaw" or "mild facial asymmetry" noted on exam does not constitute facial dysmorphism unless the clinician identifies it as dysmorphic or links it to a syndrome.
- A birthmark or single café-au-lait spot mentioned in passing does not constitute a skin pigmentation phenotype unless the clinician describes it as clinically significant, numerous, or relevant to a diagnosis.
- Height at the 15th percentile is below average but within normal range and does not qualify as short stature. Height at or below the 3rd percentile does.
- A child speaking first words at 14 months is within the normal developmental window and does not qualify as speech delay, even if slightly later than the population mean.
- Low-set or mildly prominent ears noted only under the physical exam, with no further mention, are a borderline finding and should be scored 0. If the clinician carries that finding into their diagnostic reasoning — e.g., listing "low-set, posteriorly rotated ears" among the dysmorphic features supporting a suspected syndrome in the assessment — score 1. Marked microtia or a clearly malformed pinna is scored 1 regardless of where it is documented.
- A QTc interval must be read against age-appropriate norms before it is scored as a long-QT or arrhythmia phenotype: neonates and infants normally have a longer QTc than older children, and most borderline neonatal prolongation resolves on follow-up. A single borderline QTc — roughly 450-470 ms in an infant, or 431-450 ms in an older child — should be scored 0 unless the note documents that the prolongation persisted on a repeat ECG or that long QT was diagnosed. A clearly prolonged QTc (>=500 ms at any age) or an explicit long-QT/LQTS diagnosis is scored 1 regardless of whether a history is given.
- Speech/language delay is judged against CDC milestone ages (the age by which ~75% of children show the skill): no first words besides "mama"/"dada" by 18 months, fewer than three words by 18 months, no two-word combinations (e.g., "more milk") by 24 months, or fewer than ~50 words by 30 months each meets the threshold for delay. A child saying first words at 16 months, or combining two words by 22 months, is within the normal window and does NOT qualify.
- Gross motor delay is judged against CDC/WHO milestone ages: not sitting without support by 9 months, or not walking independently ("without holding on to anyone or anything") by 18 months — the upper bound of the WHO walking window — meets the threshold. A child walking at 13-15 months is within the normal window and does NOT qualify. Absence of crawling is NOT a delay, as crawling is not a CDC milestone.
- Fine motor delay is judged against CDC milestone ages: no pincer grasp (picking up a small object between thumb and forefinger) by 12 months, or inability to hold an object in one hand while manipulating it with the other by 24 months, meets the threshold. An immature raking grasp at 9 months is age-appropriate and does NOT qualify.

PHENOTYPES:
{phenotype_list_str}

EXAMPLE OUTPUT FORMAT:
{{"Phenotypes": {{"Prematurity": {{"score": 0, "justification": "The note does not mention gestational age or premature birth.", "quote": ""}}, "Seizures": {{"score": 1, "justification": "The note documents a history of generalized tonic-clonic seizures beginning at age 3.", "quote": "Patient experienced generalized tonic-clonic seizures starting at age 3, currently managed with levetiracetam."}}, ...}}}}"""

OUTPUT_PATH = "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun6.json"
SAVE_EVERY = 10
saved_count = 0


results_claude_full_rerun6 = []

total_input_tokens = 0
total_output_tokens = 0
saved_count = 0

def save_progress():
    pd.DataFrame(results_claude_full_rerun6).to_json(OUTPUT_PATH, orient="records", indent=2)

for idx, row in rerun6.iterrows():
    subject_id = row['Subject ID']
    note_text = row['NOTE_TEXT'].strip()
    user_message = f"Clinical note for patient {subject_id}:\n\n{note_text}"
    
    max_retries = 10
    parsed = False
    
    for attempt in range(max_retries):
        with client.messages.stream(
            model="claude-sonnet-4-6",
            max_tokens=128000,
            system=system_prompt,
            messages=[{"role": "user", "content": user_message}],
            temperature=0
        ) as stream:
            response = stream.get_final_message()
        
        print(f"{subject_id} (attempt {attempt+1}) — input: {response.usage.input_tokens:,} | output: {response.usage.output_tokens:,} | stop: {response.stop_reason}")
        
        total_input_tokens += response.usage.input_tokens
        total_output_tokens += response.usage.output_tokens
        
        try:
            raw_text = response.content[0].text
            clean_text = raw_text.strip()
            if clean_text.startswith("```"):
                clean_text = clean_text.split("```")[1]
                if clean_text.startswith("json"):
                    clean_text = clean_text[4:]
                clean_text = clean_text.strip()
            
            output = json.loads(clean_text)
            phenos = output.get("Phenotypes", {})
            for pheno_name, details in phenos.items():
                results_claude_full_rerun6.append({
                    "Subject ID": subject_id,
                    "Phenotype": pheno_name,
                    "Score": details.get("score", 0),
                    "Justification": details.get("justification", ""),
                    "Quote": details.get("quote", "")
                })
            parsed = True
            break
        
        except (json.JSONDecodeError, Exception) as e:
            print(f"  Attempt {attempt+1} failed for {subject_id}: {type(e).__name__}: {e}")
            if attempt < max_retries - 1:
                print(f"  Retrying in 2 seconds...")
                time.sleep(2)
            else:
                print(f"  All {max_retries} attempts failed for {subject_id}, recording PARSE ERROR")
                results_claude_full_rerun6.append({"Subject ID": subject_id, "Phenotype": "PARSE ERROR"})
    
    time.sleep(1)  # small delay between notes
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} notes...")

    saved_count += 1
    if saved_count % SAVE_EVERY == 0:
        save_progress()
        print(f"Processed and saved {saved_count} notes...")

# final save after the loop
save_progress()
print(f"\nInput tokens: {total_input_tokens:,}")
print(f"Output tokens: {total_output_tokens:,}")
print(f"Total tokens: {total_input_tokens + total_output_tokens:,}")

results_claude_full_rerun6_df = pd.DataFrame(results_claude_full_rerun6)
print(f"Total rows: {len(results_claude_full_rerun6_df)}")
print(f"Subjects: {results_claude_full_rerun6_df['Subject ID'].nunique()}")

results_claude_full_rerun6_df.to_json(
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun6.json",
    orient="records", indent=2
)
print(f"Saved {len(results_claude_full_rerun6_df)} rows")
display(results_claude_full_rerun6_df)

Input tokens: 409,420
Output tokens: 337,350
Total tokens: 746,770
Total rows: 129
Subjects: 5
Saved 129 rows

In [0]:
results_claude_full_rerun6_df = pd.read_json("/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun6.json", orient="records")
display(results_claude_full_rerun6_df)
subjects_incomplete = (
    results_claude_full_rerun6_df.groupby('Subject ID')['Phenotype']
    .count()
    .loc[lambda x: x < 125]
    .reset_index()
    .rename(columns={'Phenotype': 'Phenotype Count'})
)

print(f"Subjects with fewer than 125 phenotypes scored: {len(subjects_incomplete)}")
display(subjects_incomplete)

#for subject_id in subjects_incomplete['Subject ID']:
 #   print(f"\nSubject ID: {subject_id}")
 #   display(results_claude_full_rerun5_df[results_claude_full_rerun5_df['Subject ID'] == subject_id])
rerun7 = notes_enrolled4[notes_enrolled4['Subject ID'].isin(subjects_incomplete['Subject ID'])].copy()
rerun7['NOTE_TEXT'] = rerun7['NOTE_TEXT'].str.strip()
display(rerun7)

In [0]:
# phenotype-specific justification
import json

phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)


phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)

system_prompt = f"""You are a world class board-certified medical geneticist performing a structured chart review.

TASK: Read the provided clinical note and determine whether each phenotype below is present (1) or absent (0) for the patient.

RULES:
- Output ONLY a valid JSON object and nothing else.
- The JSON must contain exactly one key: "Phenotypes".
- "Phenotypes": An object where each key is a phenotype name exactly as listed below, and each value is an object with three keys:
  - "score": 0 or 1
  - "justification": Why THIS SPECIFIC phenotype is present or absent. Do not summarize the entire note. Explain what evidence in the note directly relates to this phenotype, or state that no relevant evidence was found.
  - "quote": The exact word-for-word excerpt from the note that supports your decision for THIS SPECIFIC phenotype. Copy the text exactly as written in the note. Use "" if no relevant text exists.
- Each phenotype's justification and quote must pertain ONLY to that phenotype. Do not reuse the same justification or quote across multiple phenotypes unless the exact same sentence genuinely supports both.

SCORING CRITERIA:
- Use 1 ONLY if the note provides clear evidence that the patient HAS the phenotype as a clinically significant finding — a confirmed diagnosis, an objective measurement that meets a clinical threshold for pathology, or symptoms explicitly identified as pathological by the clinician.
- Use 0 if the note does not mention the phenotype, explicitly rules it out, there is insufficient evidence, or a reported value does not meet the accepted clinical threshold for that phenotype.
- If the note mentions a phenotype was historically present but has since resolved, still code as 1.
- Score based ONLY on findings documented for THIS patient. If the note describes features generally associated with a genotype, syndrome, or condition but does not confirm that this specific patient exhibits them, do not score those features as present.

CONTINUOUS AND QUANTITATIVE FEATURES:
- For any phenotype involving a measurable value, do not score based on being merely non-average.
- Score 1 ONLY if the value meets or exceeds the accepted clinical threshold for that phenotype based on CDC and WHO guidelines. For growth parameters, this generally means at or beyond the 3rd/97th percentile (±2 SD) for age and sex.
- If the note provides a raw value with the patient's age and sex, assess whether it crosses the clinical threshold. If the note already states a percentile, use that directly.
- If a value crosses the clinical threshold only marginally — sitting just past the cutoff or hovering at the borderline — do NOT score it 1 on the strength of a single reading. In that borderline case, score 1 only if the note documents a history of the finding: it has been observed on more than one occasion, or is described as persistent, recurrent, chronic, or longstanding. A lone borderline reading with no such documented history should be scored 0. Values that fall clearly and substantially beyond the threshold remain 1 regardless of whether a prior history is noted.
- A value that is below or above average but still within the normal range does NOT qualify as a phenotype. A developmental milestone that is slightly late but within the accepted normal window does NOT qualify as delayed.
- State the threshold you applied in your justification so your reasoning is transparent.

DETERMINING AGE AT A DEVELOPMENTAL MILESTONE:
- Before scoring any developmental delay, establish the patient's age at the milestone, then compare it to that milestone's threshold age.
- If the note records when the milestone occurred, use that. When it gives a date rather than an age, derive the age at the milestone from the age at the visit and the interval between the milestone date and the visit date: age at milestone = age at visit minus the time from the milestone date to the visit date. For example, if the patient is 24 months old at the visit and the milestone occurred 6 months before the visit, the age at the milestone is 18 months.
- If no date or age for the milestone is recorded, assume the age at the milestone is the patient's age at the visit. Apply this visit-age assumption to a milestone the child has not yet met; if a milestone has clearly been achieved and only its timing is unknown, score the delay 0, since you cannot establish it was reached late.
- State in your justification which age you used and whether it came from a recorded milestone date or from the visit age.

AVOIDING OVER-SCORING:
- Do NOT score normal human variation as a phenotype. Mild or subtle physical differences are normal variants unless the clinician explicitly flags them as dysmorphic or clinically significant.
- Do NOT score common benign findings as phenotypes unless the clinician identifies the finding as clinically significant, extensive, or relevant to a diagnosis.
- Qualifiers like "mild," "slight," "subtle," or "minor" should raise your threshold — these findings should only be scored as present if the clinician explicitly connects them to a diagnosis, syndrome, or clinical concern.
- Conversely, if the clinician carries a physical observation into their diagnostic reasoning — naming it in the assessment, impression, clinical concern, or differential — score it 1, even when it was first noted only in passing during the exam. The clinician tying a finding to their diagnostic thinking is itself the signal that it is clinically significant.
- When in doubt, score 0. It is better to miss a borderline finding than to falsely attribute a phenotype based on normal variation.

FACIAL DYSMORPHISM AND EAR MALFORMATION:
- Score facial dysmorphism or ear malformation only when the finding is clinically relevant. Many facial and ear characteristics reflect normal individual variation and are not, on their own, diagnostically meaningful.
- A dysmorphism or malformation that the clinician states plainly as clinically significant, or names as a specific recognized anomaly (e.g., microtia, cleft lip or palate, coloboma), is scored 1 regardless of where in the note it appears. 
- For borderline or subtle facial or ear findings, score 1 ONLY if the clinician acknowledges the dysmorphism or malformation as part of their diagnostic reasoning — in the assessment, impression, clinical concern, or differential — rather than recording it merely as an isolated observation in the physical exam. A borderline feature noted only on physical exam, without being connected to the clinician's diagnostic thinking, should be scored 0.

NON-EXHAUSTIVE EXAMPLES OF THESE PRINCIPLES:
The following illustrate how the above rules apply in practice. These are examples only — apply the same reasoning to ALL phenotypes, not just those mentioned here.
- A "slightly recessed jaw" or "mild facial asymmetry" noted on exam does not constitute facial dysmorphism unless the clinician identifies it as dysmorphic or links it to a syndrome.
- A birthmark or single café-au-lait spot mentioned in passing does not constitute a skin pigmentation phenotype unless the clinician describes it as clinically significant, numerous, or relevant to a diagnosis.
- Height at the 15th percentile is below average but within normal range and does not qualify as short stature. Height at or below the 3rd percentile does.
- A child speaking first words at 14 months is within the normal developmental window and does not qualify as speech delay, even if slightly later than the population mean.
- Low-set or mildly prominent ears noted only under the physical exam, with no further mention, are a borderline finding and should be scored 0. If the clinician carries that finding into their diagnostic reasoning — e.g., listing "low-set, posteriorly rotated ears" among the dysmorphic features supporting a suspected syndrome in the assessment — score 1. Marked microtia or a clearly malformed pinna is scored 1 regardless of where it is documented.
- A QTc interval must be read against age-appropriate norms before it is scored as a long-QT or arrhythmia phenotype: neonates and infants normally have a longer QTc than older children, and most borderline neonatal prolongation resolves on follow-up. A single borderline QTc — roughly 450-470 ms in an infant, or 431-450 ms in an older child — should be scored 0 unless the note documents that the prolongation persisted on a repeat ECG or that long QT was diagnosed. A clearly prolonged QTc (>=500 ms at any age) or an explicit long-QT/LQTS diagnosis is scored 1 regardless of whether a history is given.
- Speech/language delay is judged against CDC milestone ages (the age by which ~75% of children show the skill): no first words besides "mama"/"dada" by 18 months, fewer than three words by 18 months, no two-word combinations (e.g., "more milk") by 24 months, or fewer than ~50 words by 30 months each meets the threshold for delay. A child saying first words at 16 months, or combining two words by 22 months, is within the normal window and does NOT qualify.
- Gross motor delay is judged against CDC/WHO milestone ages: not sitting without support by 9 months, or not walking independently ("without holding on to anyone or anything") by 18 months — the upper bound of the WHO walking window — meets the threshold. A child walking at 13-15 months is within the normal window and does NOT qualify. Absence of crawling is NOT a delay, as crawling is not a CDC milestone.
- Fine motor delay is judged against CDC milestone ages: no pincer grasp (picking up a small object between thumb and forefinger) by 12 months, or inability to hold an object in one hand while manipulating it with the other by 24 months, meets the threshold. An immature raking grasp at 9 months is age-appropriate and does NOT qualify.

PHENOTYPES:
{phenotype_list_str}

EXAMPLE OUTPUT FORMAT:
{{"Phenotypes": {{"Prematurity": {{"score": 0, "justification": "The note does not mention gestational age or premature birth.", "quote": ""}}, "Seizures": {{"score": 1, "justification": "The note documents a history of generalized tonic-clonic seizures beginning at age 3.", "quote": "Patient experienced generalized tonic-clonic seizures starting at age 3, currently managed with levetiracetam."}}, ...}}}}"""

OUTPUT_PATH = "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun7.json"
SAVE_EVERY = 10
saved_count = 0


results_claude_full_rerun7 = []

total_input_tokens = 0
total_output_tokens = 0
saved_count = 0

def save_progress():
    pd.DataFrame(results_claude_full_rerun7).to_json(OUTPUT_PATH, orient="records", indent=2)

for idx, row in rerun7.iterrows():
    subject_id = row['Subject ID']
    note_text = row['NOTE_TEXT'].strip()
    user_message = f"Clinical note for patient {subject_id}:\n\n{note_text}"
    
    max_retries = 10
    parsed = False
    
    for attempt in range(max_retries):
        with client.messages.stream(
            model="claude-sonnet-4-6",
            max_tokens=128000,
            system=system_prompt,
            messages=[{"role": "user", "content": user_message}],
            temperature=0
        ) as stream:
            response = stream.get_final_message()
        
        print(f"{subject_id} (attempt {attempt+1}) — input: {response.usage.input_tokens:,} | output: {response.usage.output_tokens:,} | stop: {response.stop_reason}")
        
        total_input_tokens += response.usage.input_tokens
        total_output_tokens += response.usage.output_tokens
        
        try:
            raw_text = response.content[0].text
            clean_text = raw_text.strip()
            if clean_text.startswith("```"):
                clean_text = clean_text.split("```")[1]
                if clean_text.startswith("json"):
                    clean_text = clean_text[4:]
                clean_text = clean_text.strip()
            
            output = json.loads(clean_text)
            phenos = output.get("Phenotypes", {})
            for pheno_name, details in phenos.items():
                results_claude_full_rerun7.append({
                    "Subject ID": subject_id,
                    "Phenotype": pheno_name,
                    "Score": details.get("score", 0),
                    "Justification": details.get("justification", ""),
                    "Quote": details.get("quote", "")
                })
            parsed = True
            break
        
        except (json.JSONDecodeError, Exception) as e:
            print(f"  Attempt {attempt+1} failed for {subject_id}: {type(e).__name__}: {e}")
            if attempt < max_retries - 1:
                print(f"  Retrying in 2 seconds...")
                time.sleep(2)
            else:
                print(f"  All {max_retries} attempts failed for {subject_id}, recording PARSE ERROR")
                results_claude_full_rerun7.append({"Subject ID": subject_id, "Phenotype": "PARSE ERROR"})
    
    time.sleep(1)  # small delay between notes
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} notes...")

    saved_count += 1
    if saved_count % SAVE_EVERY == 0:
        save_progress()
        print(f"Processed and saved {saved_count} notes...")

# final save after the loop
save_progress()
print(f"\nInput tokens: {total_input_tokens:,}")
print(f"Output tokens: {total_output_tokens:,}")
print(f"Total tokens: {total_input_tokens + total_output_tokens:,}")

results_claude_full_rerun7_df = pd.DataFrame(results_claude_full_rerun7)
print(f"Total rows: {len(results_claude_full_rerun7_df)}")
print(f"Subjects: {results_claude_full_rerun7_df['Subject ID'].nunique()}")

results_claude_full_rerun7_df.to_json(
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun7.json",
    orient="records", indent=2
)
print(f"Saved {len(results_claude_full_rerun7_df)} rows")
display(results_claude_full_rerun7_df)

Input tokens: 215,692
Output tokens: 186,506
Total tokens: 402,198
Total rows: 252
Subjects: 4
Saved 252 rows

In [0]:
results_claude_full_rerun7_df = pd.read_json("/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun7.json", orient="records")
display(results_claude_full_rerun7_df)
subjects_incomplete = (
    results_claude_full_rerun7_df.groupby('Subject ID')['Phenotype']
    .count()
    .loc[lambda x: x < 125]
    .reset_index()
    .rename(columns={'Phenotype': 'Phenotype Count'})
)

print(f"Subjects with fewer than 125 phenotypes scored: {len(subjects_incomplete)}")
display(subjects_incomplete)

rerun8 = notes_enrolled4[notes_enrolled4['Subject ID'].isin(subjects_incomplete['Subject ID'])].copy()
rerun8['NOTE_TEXT'] = rerun8['NOTE_TEXT'].str.strip()
display(rerun8)

In [0]:
# phenotype-specific justification
import json

phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)


phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)

system_prompt = f"""You are a world class board-certified medical geneticist performing a structured chart review.

TASK: Read the provided clinical note and determine whether each phenotype below is present (1) or absent (0) for the patient.

RULES:
- Output ONLY a valid JSON object and nothing else.
- The JSON must contain exactly one key: "Phenotypes".
- "Phenotypes": An object where each key is a phenotype name exactly as listed below, and each value is an object with three keys:
  - "score": 0 or 1
  - "justification": Why THIS SPECIFIC phenotype is present or absent. Do not summarize the entire note. Explain what evidence in the note directly relates to this phenotype, or state that no relevant evidence was found.
  - "quote": The exact word-for-word excerpt from the note that supports your decision for THIS SPECIFIC phenotype. Copy the text exactly as written in the note. Use "" if no relevant text exists.
- Each phenotype's justification and quote must pertain ONLY to that phenotype. Do not reuse the same justification or quote across multiple phenotypes unless the exact same sentence genuinely supports both.

SCORING CRITERIA:
- Use 1 ONLY if the note provides clear evidence that the patient HAS the phenotype as a clinically significant finding — a confirmed diagnosis, an objective measurement that meets a clinical threshold for pathology, or symptoms explicitly identified as pathological by the clinician.
- Use 0 if the note does not mention the phenotype, explicitly rules it out, there is insufficient evidence, or a reported value does not meet the accepted clinical threshold for that phenotype.
- If the note mentions a phenotype was historically present but has since resolved, still code as 1.
- Score based ONLY on findings documented for THIS patient. If the note describes features generally associated with a genotype, syndrome, or condition but does not confirm that this specific patient exhibits them, do not score those features as present.

CONTINUOUS AND QUANTITATIVE FEATURES:
- For any phenotype involving a measurable value, do not score based on being merely non-average.
- Score 1 ONLY if the value meets or exceeds the accepted clinical threshold for that phenotype based on CDC and WHO guidelines. For growth parameters, this generally means at or beyond the 3rd/97th percentile (±2 SD) for age and sex.
- If the note provides a raw value with the patient's age and sex, assess whether it crosses the clinical threshold. If the note already states a percentile, use that directly.
- If a value crosses the clinical threshold only marginally — sitting just past the cutoff or hovering at the borderline — do NOT score it 1 on the strength of a single reading. In that borderline case, score 1 only if the note documents a history of the finding: it has been observed on more than one occasion, or is described as persistent, recurrent, chronic, or longstanding. A lone borderline reading with no such documented history should be scored 0. Values that fall clearly and substantially beyond the threshold remain 1 regardless of whether a prior history is noted.
- A value that is below or above average but still within the normal range does NOT qualify as a phenotype. A developmental milestone that is slightly late but within the accepted normal window does NOT qualify as delayed.
- State the threshold you applied in your justification so your reasoning is transparent.

DETERMINING AGE AT A DEVELOPMENTAL MILESTONE:
- Before scoring any developmental delay, establish the patient's age at the milestone, then compare it to that milestone's threshold age.
- If the note records when the milestone occurred, use that. When it gives a date rather than an age, derive the age at the milestone from the age at the visit and the interval between the milestone date and the visit date: age at milestone = age at visit minus the time from the milestone date to the visit date. For example, if the patient is 24 months old at the visit and the milestone occurred 6 months before the visit, the age at the milestone is 18 months.
- If no date or age for the milestone is recorded, assume the age at the milestone is the patient's age at the visit. Apply this visit-age assumption to a milestone the child has not yet met; if a milestone has clearly been achieved and only its timing is unknown, score the delay 0, since you cannot establish it was reached late.
- State in your justification which age you used and whether it came from a recorded milestone date or from the visit age.

AVOIDING OVER-SCORING:
- Do NOT score normal human variation as a phenotype. Mild or subtle physical differences are normal variants unless the clinician explicitly flags them as dysmorphic or clinically significant.
- Do NOT score common benign findings as phenotypes unless the clinician identifies the finding as clinically significant, extensive, or relevant to a diagnosis.
- Qualifiers like "mild," "slight," "subtle," or "minor" should raise your threshold — these findings should only be scored as present if the clinician explicitly connects them to a diagnosis, syndrome, or clinical concern.
- Conversely, if the clinician carries a physical observation into their diagnostic reasoning — naming it in the assessment, impression, clinical concern, or differential — score it 1, even when it was first noted only in passing during the exam. The clinician tying a finding to their diagnostic thinking is itself the signal that it is clinically significant.
- When in doubt, score 0. It is better to miss a borderline finding than to falsely attribute a phenotype based on normal variation.

FACIAL DYSMORPHISM AND EAR MALFORMATION:
- Score facial dysmorphism or ear malformation only when the finding is clinically relevant. Many facial and ear characteristics reflect normal individual variation and are not, on their own, diagnostically meaningful.
- A dysmorphism or malformation that the clinician states plainly as clinically significant, or names as a specific recognized anomaly (e.g., microtia, cleft lip or palate, coloboma), is scored 1 regardless of where in the note it appears. 
- For borderline or subtle facial or ear findings, score 1 ONLY if the clinician acknowledges the dysmorphism or malformation as part of their diagnostic reasoning — in the assessment, impression, clinical concern, or differential — rather than recording it merely as an isolated observation in the physical exam. A borderline feature noted only on physical exam, without being connected to the clinician's diagnostic thinking, should be scored 0.

NON-EXHAUSTIVE EXAMPLES OF THESE PRINCIPLES:
The following illustrate how the above rules apply in practice. These are examples only — apply the same reasoning to ALL phenotypes, not just those mentioned here.
- A "slightly recessed jaw" or "mild facial asymmetry" noted on exam does not constitute facial dysmorphism unless the clinician identifies it as dysmorphic or links it to a syndrome.
- A birthmark or single café-au-lait spot mentioned in passing does not constitute a skin pigmentation phenotype unless the clinician describes it as clinically significant, numerous, or relevant to a diagnosis.
- Height at the 15th percentile is below average but within normal range and does not qualify as short stature. Height at or below the 3rd percentile does.
- A child speaking first words at 14 months is within the normal developmental window and does not qualify as speech delay, even if slightly later than the population mean.
- Low-set or mildly prominent ears noted only under the physical exam, with no further mention, are a borderline finding and should be scored 0. If the clinician carries that finding into their diagnostic reasoning — e.g., listing "low-set, posteriorly rotated ears" among the dysmorphic features supporting a suspected syndrome in the assessment — score 1. Marked microtia or a clearly malformed pinna is scored 1 regardless of where it is documented.
- A QTc interval must be read against age-appropriate norms before it is scored as a long-QT or arrhythmia phenotype: neonates and infants normally have a longer QTc than older children, and most borderline neonatal prolongation resolves on follow-up. A single borderline QTc — roughly 450-470 ms in an infant, or 431-450 ms in an older child — should be scored 0 unless the note documents that the prolongation persisted on a repeat ECG or that long QT was diagnosed. A clearly prolonged QTc (>=500 ms at any age) or an explicit long-QT/LQTS diagnosis is scored 1 regardless of whether a history is given.
- Speech/language delay is judged against CDC milestone ages (the age by which ~75% of children show the skill): no first words besides "mama"/"dada" by 18 months, fewer than three words by 18 months, no two-word combinations (e.g., "more milk") by 24 months, or fewer than ~50 words by 30 months each meets the threshold for delay. A child saying first words at 16 months, or combining two words by 22 months, is within the normal window and does NOT qualify.
- Gross motor delay is judged against CDC/WHO milestone ages: not sitting without support by 9 months, or not walking independently ("without holding on to anyone or anything") by 18 months — the upper bound of the WHO walking window — meets the threshold. A child walking at 13-15 months is within the normal window and does NOT qualify. Absence of crawling is NOT a delay, as crawling is not a CDC milestone.
- Fine motor delay is judged against CDC milestone ages: no pincer grasp (picking up a small object between thumb and forefinger) by 12 months, or inability to hold an object in one hand while manipulating it with the other by 24 months, meets the threshold. An immature raking grasp at 9 months is age-appropriate and does NOT qualify.

PHENOTYPES:
{phenotype_list_str}

EXAMPLE OUTPUT FORMAT:
{{"Phenotypes": {{"Prematurity": {{"score": 0, "justification": "The note does not mention gestational age or premature birth.", "quote": ""}}, "Seizures": {{"score": 1, "justification": "The note documents a history of generalized tonic-clonic seizures beginning at age 3.", "quote": "Patient experienced generalized tonic-clonic seizures starting at age 3, currently managed with levetiracetam."}}, ...}}}}"""

OUTPUT_PATH = "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun8.json"
SAVE_EVERY = 10
saved_count = 0


results_claude_full_rerun8 = []

total_input_tokens = 0
total_output_tokens = 0
saved_count = 0

def save_progress():
    pd.DataFrame(results_claude_full_rerun8).to_json(OUTPUT_PATH, orient="records", indent=2)

for idx, row in rerun8.iterrows():
    subject_id = row['Subject ID']
    note_text = row['NOTE_TEXT'].strip()
    user_message = f"Clinical note for patient {subject_id}:\n\n{note_text}"
    
    max_retries = 10
    parsed = False
    
    for attempt in range(max_retries):
        with client.messages.stream(
            model="claude-sonnet-4-6",
            max_tokens=128000,
            system=system_prompt,
            messages=[{"role": "user", "content": user_message}],
            temperature=0
        ) as stream:
            response = stream.get_final_message()
        
        print(f"{subject_id} (attempt {attempt+1}) — input: {response.usage.input_tokens:,} | output: {response.usage.output_tokens:,} | stop: {response.stop_reason}")
        
        total_input_tokens += response.usage.input_tokens
        total_output_tokens += response.usage.output_tokens
        
        try:
            raw_text = response.content[0].text
            clean_text = raw_text.strip()
            if clean_text.startswith("```"):
                clean_text = clean_text.split("```")[1]
                if clean_text.startswith("json"):
                    clean_text = clean_text[4:]
                clean_text = clean_text.strip()
            
            output = json.loads(clean_text)
            phenos = output.get("Phenotypes", {})
            for pheno_name, details in phenos.items():
                results_claude_full_rerun8.append({
                    "Subject ID": subject_id,
                    "Phenotype": pheno_name,
                    "Score": details.get("score", 0),
                    "Justification": details.get("justification", ""),
                    "Quote": details.get("quote", "")
                })
            parsed = True
            break
        
        except (json.JSONDecodeError, Exception) as e:
            print(f"  Attempt {attempt+1} failed for {subject_id}: {type(e).__name__}: {e}")
            if attempt < max_retries - 1:
                print(f"  Retrying in 2 seconds...")
                time.sleep(2)
            else:
                print(f"  All {max_retries} attempts failed for {subject_id}, recording PARSE ERROR")
                results_claude_full_rerun8.append({"Subject ID": subject_id, "Phenotype": "PARSE ERROR"})
    
    time.sleep(1)  # small delay between notes
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} notes...")

    saved_count += 1
    if saved_count % SAVE_EVERY == 0:
        save_progress()
        print(f"Processed and saved {saved_count} notes...")

# final save after the loop
save_progress()
print(f"\nInput tokens: {total_input_tokens:,}")
print(f"Output tokens: {total_output_tokens:,}")
print(f"Total tokens: {total_input_tokens + total_output_tokens:,}")

results_claude_full_rerun8_df = pd.DataFrame(results_claude_full_rerun8)
print(f"Total rows: {len(results_claude_full_rerun8_df)}")
print(f"Subjects: {results_claude_full_rerun8_df['Subject ID'].nunique()}")

results_claude_full_rerun8_df.to_json(
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun8.json",
    orient="records", indent=2
)
print(f"Saved {len(results_claude_full_rerun8_df)} rows")
display(results_claude_full_rerun8_df)

Input tokens: 63,838
Output tokens: 64,676
Total tokens: 128,514
Total rows: 126
Subjects: 2
Saved 126 rows

In [0]:
results_claude_full_rerun8_df = pd.read_json("/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun8.json", orient="records")
display(results_claude_full_rerun8_df)
subjects_incomplete = (
    results_claude_full_rerun8_df.groupby('Subject ID')['Phenotype']
    .count()
    .loc[lambda x: x < 125]
    .reset_index()
    .rename(columns={'Phenotype': 'Phenotype Count'})
)

print(f"Subjects with fewer than 125 phenotypes scored: {len(subjects_incomplete)}")
display(subjects_incomplete)

#for subject_id in subjects_incomplete['Subject ID']:
 #   print(f"\nSubject ID: {subject_id}")
 #   display(results_claude_full_rerun5_df[results_claude_full_rerun5_df['Subject ID'] == subject_id])
rerun9 = notes_enrolled4[notes_enrolled4['Subject ID'].isin(subjects_incomplete['Subject ID'])].copy()
rerun9['NOTE_TEXT'] = rerun9['NOTE_TEXT'].str.strip()
display(rerun9)

In [0]:
for idx, row in rerun9.iterrows():
    note = row['NOTE_TEXT']
    subject_id = row['Subject ID']
    
    # Length
    print(f"{subject_id}: {len(note)} chars")
    
    # Quotation marks
    print(f"  Double quotes: {note.count(chr(34))}")
    print(f"  Curly quotes: {note.count(chr(8220)) + note.count(chr(8221))}")
    
    # Non-ASCII characters
    non_ascii = [c for c in note if ord(c) > 127]
    print(f"  Non-ASCII chars: {len(non_ascii)} — {set(non_ascii) if non_ascii else 'none'}")
    
    # Unusual whitespace
    print(f"  Tabs: {note.count(chr(9))}")
    print(f"  Carriage returns: {note.count(chr(13))}")
    
    print("---")

In [0]:
for idx, row in notes_enrolled4.iterrows():
    note = row['NOTE_TEXT']
    subject_id = row['Subject ID']
    
    non_ascii = [c for c in note if ord(c) > 127]
    double_quotes = note.count('"')
    curly_quotes = note.count('\u201c') + note.count('\u201d')
    
    if non_ascii or double_quotes > 0 or curly_quotes > 0:
        print(f"{subject_id}: {len(note)} chars")
        print(f"  Double quotes: {double_quotes}")
        print(f"  Curly quotes: {curly_quotes}")
        print(f"  Non-ASCII chars: {len(non_ascii)} — {set(non_ascii)}")
        print("---")

In [0]:
# phenotype-specific justification
import json

phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)


phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)

system_prompt = f"""You are a world class board-certified medical geneticist performing a structured chart review.

TASK: Read the provided clinical note and determine whether each phenotype below is present (1) or absent (0) for the patient.

RULES:
- Output ONLY a valid JSON object and nothing else.
- The JSON must contain exactly one key: "Phenotypes".
- "Phenotypes": An object where each key is a phenotype name exactly as listed below, and each value is an object with three keys:
  - "score": 0 or 1
  - "justification": Why THIS SPECIFIC phenotype is present or absent. Do not summarize the entire note. Explain what evidence in the note directly relates to this phenotype, or state that no relevant evidence was found.
  - "quote": The exact word-for-word excerpt from the note that supports your decision for THIS SPECIFIC phenotype. Copy the text exactly as written in the note. Use "" if no relevant text exists.
- Each phenotype's justification and quote must pertain ONLY to that phenotype. Do not reuse the same justification or quote across multiple phenotypes unless the exact same sentence genuinely supports both.

SCORING CRITERIA:
- Use 1 ONLY if the note provides clear evidence that the patient HAS the phenotype as a clinically significant finding — a confirmed diagnosis, an objective measurement that meets a clinical threshold for pathology, or symptoms explicitly identified as pathological by the clinician.
- Use 0 if the note does not mention the phenotype, explicitly rules it out, there is insufficient evidence, or a reported value does not meet the accepted clinical threshold for that phenotype.
- If the note mentions a phenotype was historically present but has since resolved, still code as 1.
- Score based ONLY on findings documented for THIS patient. If the note describes features generally associated with a genotype, syndrome, or condition but does not confirm that this specific patient exhibits them, do not score those features as present.

CONTINUOUS AND QUANTITATIVE FEATURES:
- For any phenotype involving a measurable value, do not score based on being merely non-average.
- Score 1 ONLY if the value meets or exceeds the accepted clinical threshold for that phenotype based on CDC and WHO guidelines. For growth parameters, this generally means at or beyond the 3rd/97th percentile (±2 SD) for age and sex.
- If the note provides a raw value with the patient's age and sex, assess whether it crosses the clinical threshold. If the note already states a percentile, use that directly.
- If a value crosses the clinical threshold only marginally — sitting just past the cutoff or hovering at the borderline — do NOT score it 1 on the strength of a single reading. In that borderline case, score 1 only if the note documents a history of the finding: it has been observed on more than one occasion, or is described as persistent, recurrent, chronic, or longstanding. A lone borderline reading with no such documented history should be scored 0. Values that fall clearly and substantially beyond the threshold remain 1 regardless of whether a prior history is noted.
- A value that is below or above average but still within the normal range does NOT qualify as a phenotype. A developmental milestone that is slightly late but within the accepted normal window does NOT qualify as delayed.
- State the threshold you applied in your justification so your reasoning is transparent.

DETERMINING AGE AT A DEVELOPMENTAL MILESTONE:
- Before scoring any developmental delay, establish the patient's age at the milestone, then compare it to that milestone's threshold age.
- If the note records when the milestone occurred, use that. When it gives a date rather than an age, derive the age at the milestone from the age at the visit and the interval between the milestone date and the visit date: age at milestone = age at visit minus the time from the milestone date to the visit date. For example, if the patient is 24 months old at the visit and the milestone occurred 6 months before the visit, the age at the milestone is 18 months.
- If no date or age for the milestone is recorded, assume the age at the milestone is the patient's age at the visit. Apply this visit-age assumption to a milestone the child has not yet met; if a milestone has clearly been achieved and only its timing is unknown, score the delay 0, since you cannot establish it was reached late.
- State in your justification which age you used and whether it came from a recorded milestone date or from the visit age.

AVOIDING OVER-SCORING:
- Do NOT score normal human variation as a phenotype. Mild or subtle physical differences are normal variants unless the clinician explicitly flags them as dysmorphic or clinically significant.
- Do NOT score common benign findings as phenotypes unless the clinician identifies the finding as clinically significant, extensive, or relevant to a diagnosis.
- Qualifiers like "mild," "slight," "subtle," or "minor" should raise your threshold — these findings should only be scored as present if the clinician explicitly connects them to a diagnosis, syndrome, or clinical concern.
- Conversely, if the clinician carries a physical observation into their diagnostic reasoning — naming it in the assessment, impression, clinical concern, or differential — score it 1, even when it was first noted only in passing during the exam. The clinician tying a finding to their diagnostic thinking is itself the signal that it is clinically significant.
- When in doubt, score 0. It is better to miss a borderline finding than to falsely attribute a phenotype based on normal variation.

FACIAL DYSMORPHISM AND EAR MALFORMATION:
- Score facial dysmorphism or ear malformation only when the finding is clinically relevant. Many facial and ear characteristics reflect normal individual variation and are not, on their own, diagnostically meaningful.
- A dysmorphism or malformation that the clinician states plainly as clinically significant, or names as a specific recognized anomaly (e.g., microtia, cleft lip or palate, coloboma), is scored 1 regardless of where in the note it appears. 
- For borderline or subtle facial or ear findings, score 1 ONLY if the clinician acknowledges the dysmorphism or malformation as part of their diagnostic reasoning — in the assessment, impression, clinical concern, or differential — rather than recording it merely as an isolated observation in the physical exam. A borderline feature noted only on physical exam, without being connected to the clinician's diagnostic thinking, should be scored 0.

NON-EXHAUSTIVE EXAMPLES OF THESE PRINCIPLES:
The following illustrate how the above rules apply in practice. These are examples only — apply the same reasoning to ALL phenotypes, not just those mentioned here.
- A "slightly recessed jaw" or "mild facial asymmetry" noted on exam does not constitute facial dysmorphism unless the clinician identifies it as dysmorphic or links it to a syndrome.
- A birthmark or single café-au-lait spot mentioned in passing does not constitute a skin pigmentation phenotype unless the clinician describes it as clinically significant, numerous, or relevant to a diagnosis.
- Height at the 15th percentile is below average but within normal range and does not qualify as short stature. Height at or below the 3rd percentile does.
- A child speaking first words at 14 months is within the normal developmental window and does not qualify as speech delay, even if slightly later than the population mean.
- Low-set or mildly prominent ears noted only under the physical exam, with no further mention, are a borderline finding and should be scored 0. If the clinician carries that finding into their diagnostic reasoning — e.g., listing "low-set, posteriorly rotated ears" among the dysmorphic features supporting a suspected syndrome in the assessment — score 1. Marked microtia or a clearly malformed pinna is scored 1 regardless of where it is documented.
- A QTc interval must be read against age-appropriate norms before it is scored as a long-QT or arrhythmia phenotype: neonates and infants normally have a longer QTc than older children, and most borderline neonatal prolongation resolves on follow-up. A single borderline QTc — roughly 450-470 ms in an infant, or 431-450 ms in an older child — should be scored 0 unless the note documents that the prolongation persisted on a repeat ECG or that long QT was diagnosed. A clearly prolonged QTc (>=500 ms at any age) or an explicit long-QT/LQTS diagnosis is scored 1 regardless of whether a history is given.
- Speech/language delay is judged against CDC milestone ages (the age by which ~75% of children show the skill): no first words besides "mama"/"dada" by 18 months, fewer than three words by 18 months, no two-word combinations (e.g., "more milk") by 24 months, or fewer than ~50 words by 30 months each meets the threshold for delay. A child saying first words at 16 months, or combining two words by 22 months, is within the normal window and does NOT qualify.
- Gross motor delay is judged against CDC/WHO milestone ages: not sitting without support by 9 months, or not walking independently ("without holding on to anyone or anything") by 18 months — the upper bound of the WHO walking window — meets the threshold. A child walking at 13-15 months is within the normal window and does NOT qualify. Absence of crawling is NOT a delay, as crawling is not a CDC milestone.
- Fine motor delay is judged against CDC milestone ages: no pincer grasp (picking up a small object between thumb and forefinger) by 12 months, or inability to hold an object in one hand while manipulating it with the other by 24 months, meets the threshold. An immature raking grasp at 9 months is age-appropriate and does NOT qualify.

PHENOTYPES:
{phenotype_list_str}

EXAMPLE OUTPUT FORMAT:
{{"Phenotypes": {{"Prematurity": {{"score": 0, "justification": "The note does not mention gestational age or premature birth.", "quote": ""}}, "Seizures": {{"score": 1, "justification": "The note documents a history of generalized tonic-clonic seizures beginning at age 3.", "quote": "Patient experienced generalized tonic-clonic seizures starting at age 3, currently managed with levetiracetam."}}, ...}}}}"""

OUTPUT_PATH = "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun9.json"
SAVE_EVERY = 10
saved_count = 0


results_claude_full_rerun9 = []

total_input_tokens = 0
total_output_tokens = 0
saved_count = 0

def save_progress():
    pd.DataFrame(results_claude_full_rerun9).to_json(OUTPUT_PATH, orient="records", indent=2)

for idx, row in rerun9.iterrows():
    subject_id = row['Subject ID']
    note_text = row['NOTE_TEXT'].strip()
    user_message = f"Clinical note for patient {subject_id}:\n\n{note_text}"
    
    max_retries = 20
    parsed = False
    
    for attempt in range(max_retries):
        with client.messages.stream(
            model="claude-sonnet-4-6",
            max_tokens=128000,
            system=system_prompt,
            messages=[{"role": "user", "content": user_message}],
            temperature=0
        ) as stream:
            response = stream.get_final_message()
        
        print(f"{subject_id} (attempt {attempt+1}) — input: {response.usage.input_tokens:,} | output: {response.usage.output_tokens:,} | stop: {response.stop_reason}")
        
        total_input_tokens += response.usage.input_tokens
        total_output_tokens += response.usage.output_tokens
        
        try:
            raw_text = response.content[0].text
            clean_text = raw_text.strip()
            if clean_text.startswith("```"):
                clean_text = clean_text.split("```")[1]
                if clean_text.startswith("json"):
                    clean_text = clean_text[4:]
                clean_text = clean_text.strip()
            
            output = json.loads(clean_text)
            phenos = output.get("Phenotypes", {})
            for pheno_name, details in phenos.items():
                results_claude_full_rerun9.append({
                    "Subject ID": subject_id,
                    "Phenotype": pheno_name,
                    "Score": details.get("score", 0),
                    "Justification": details.get("justification", ""),
                    "Quote": details.get("quote", "")
                })
            parsed = True
            break
        
        except (json.JSONDecodeError, Exception) as e:
            print(f"  Attempt {attempt+1} failed for {subject_id}: {type(e).__name__}: {e}")
            if attempt < max_retries - 1:
                print(f"  Retrying in 2 seconds...")
                time.sleep(2)
            else:
                print(f"  All {max_retries} attempts failed for {subject_id}, recording PARSE ERROR")
                results_claude_full_rerun9.append({"Subject ID": subject_id, "Phenotype": "PARSE ERROR"})
    
    time.sleep(1)  # small delay between notes
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} notes...")

    saved_count += 1
    if saved_count % SAVE_EVERY == 0:
        save_progress()
        print(f"Processed and saved {saved_count} notes...")

# final save after the loop
save_progress()
print(f"\nInput tokens: {total_input_tokens:,}")
print(f"Output tokens: {total_output_tokens:,}")
print(f"Total tokens: {total_input_tokens + total_output_tokens:,}")

results_claude_full_rerun9_df = pd.DataFrame(results_claude_full_rerun9)
print(f"Total rows: {len(results_claude_full_rerun9_df)}")
print(f"Subjects: {results_claude_full_rerun9_df['Subject ID'].nunique()}")

results_claude_full_rerun9_df.to_json(
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun9.json",
    orient="records", indent=2
)
print(f"Saved {len(results_claude_full_rerun9_df)} rows")
display(results_claude_full_rerun9_df)

In [0]:
results_claude_full_rerun9_df = pd.read_json("/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun9.json", orient="records")
display(results_claude_full_rerun9_df)
subjects_incomplete = (
    results_claude_full_rerun9_df.groupby('Subject ID')['Phenotype']
    .count()
    .loc[lambda x: x < 125]
    .reset_index()
    .rename(columns={'Phenotype': 'Phenotype Count'})
)

print(f"Subjects with fewer than 125 phenotypes scored: {len(subjects_incomplete)}")
display(subjects_incomplete)

#for subject_id in subjects_incomplete['Subject ID']:
 #   print(f"\nSubject ID: {subject_id}")
 #   display(results_claude_full_rerun5_df[results_claude_full_rerun5_df['Subject ID'] == subject_id])
rerun10 = notes_enrolled4[notes_enrolled4['Subject ID'].isin(subjects_incomplete['Subject ID'])].copy()
rerun10['NOTE_TEXT'] = rerun10['NOTE_TEXT'].str.strip()
display(rerun10)

In [0]:
import pandas as pd
import json

# Load main file
main_df = pd.read_json("/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full.json")

# Load all rerun files
rerun_paths = [
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun.json",
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun2.json",
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun3.json",
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun4.json",
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun5.json",
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun6.json",
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun7.json",
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun8.json",
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_rerun9.json",
]

rerun_dfs = [pd.read_json(path) for path in rerun_paths]
rerun_df = pd.concat(rerun_dfs, ignore_index=True)

# Keep only successful rows from reruns (no PARSE ERROR)
rerun_success = rerun_df[rerun_df['Phenotype'] != 'PARSE ERROR']

# Get subject IDs that were successfully parsed in reruns
rerun_subjects = rerun_success['Subject ID'].unique()

# Remove PARSE ERROR rows from main file for subjects that were fixed in reruns
main_df = main_df[~((main_df['Subject ID'].isin(rerun_subjects)) & (main_df['Phenotype'] == 'PARSE ERROR'))]

# Append successful rerun rows
final_df = pd.concat([main_df, rerun_success], ignore_index=True)

# Check results
print(f"Total rows: {len(final_df)}")
print(f"Total subjects: {final_df['Subject ID'].nunique()}")
parse_errors = final_df[final_df['Phenotype'] == 'PARSE ERROR']
print(f"Remaining parse errors: {len(parse_errors)}")
print(f"Subjects with parse errors: {parse_errors['Subject ID'].tolist()}")

# Save final file
final_df.to_json(
    "/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_final.json",
    orient="records", indent=2
)
print(f"Saved {len(final_df)} rows")